This script tool converts the scaler and ranker into an inference friendly format (JSON), moves all relevant model artifacts, writes the inference wrapper, runs some inference tests (on the function directly, the inference wrapper, & the flask API wrapper)


In [1]:
#CHANGE VERSION NUMBER IN CELL BELOW
#Define values
writeOutInferenceFile= True
moveArtifactsToInference=True
scalerUsed = False
NUM_HISTORY=10
artifact_destination_dir = '../ModelImplementationWSDK/Math_student_proficiency_model/container/'
IPfile = 'Math_ItemParameters_v5.csv'
pipeline_folder = "item_param_calc_50/"
#model_name = '01312025_noSkillExplode_v50_ModelOnlyIncompleteIP'
model_name = '01312025_noSkillExplode_v50Full_standard'

# models_v30_v33IP_moreFeatures_v33
# Define source paths and new names
source_paths = {
    'item_params': pipeline_folder + IPfile,
    
    'proficiency_model': pipeline_folder + 'proficiency_model_'+ model_name + '/xgb_model.json',
    'proficiency_scaler': pipeline_folder + 'proficiency_model_'+ model_name + '/nan_preserving_scaler_proficiency.joblib',
    
    'confidence_model': pipeline_folder + 'confidence_model_'+ model_name + '/confidence_xgb_model.json',
    'confidence_score_scaler': pipeline_folder + 'confidence_model_'+ model_name + '/percentile_rank_calculator.pkl'
}

# Define source paths and new names
# source_paths = {
#     'item_params': 'ELA_ItemParameters_4monthsv13.csv',
    
#     'proficiency_model': 'models_v30_continuousIP_moreFeatures_continuous/xgb_model.json',
#     'proficiency_scaler': 'models_v30_continuousIP_moreFeatures_continuous/nan_preserving_scaler_proficiency.joblib',
    
#     'confidence_model': 'models_v30_continuousIP_moreFeatures_continuous/confidence_xgb_model.json',
#     'confidence_score_scaler': 'models_v30_continuousIP_moreFeatures_continuous/percentile_rank_calculator.pkl'
# }


# Define new names for the files
new_names = {
    'item_params': 'item_params.csv',
    
    'proficiency_model': 'proficiency_model.json',
    'proficiency_scaler': 'proficiency_scaler.joblib',
    
    'confidence_model': 'confidence_model.json',
    'confidence_score_scaler': 'confidence_score_scaler.pkl'
}

if not scalerUsed:
    source_paths.pop('proficiency_scaler', None)
    new_names.pop('proficiency_scaler', None)


In [2]:
#Load scaler and convert it
if scalerUsed:
    if source_paths['proficiency_scaler'].rsplit('.', 1)[1]=='joblib':
        scaler_location=source_paths['proficiency_scaler']
        new_scaler_location = scaler_location.rsplit('.', 1)[0] + '.json'
        print('loading from '+str(scaler_location))

        #Define scaler class

        import json
        import numpy as np
        from sklearn.preprocessing import StandardScaler
        import joblib

        # Load the proficiency scaler
        class NanPreservingScaler:
            def __init__(self):
                self.scalers = {}
                self.column_means = {}
                self.numeric_columns = None

            def partial_fit(self, X):
                if isinstance(X, np.ndarray):
                    X = pd.DataFrame(X, columns=[f'col_{i}' for i in range(X.shape[1])])

                if self.numeric_columns is None:
                    self.numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()

                X_numeric = X[self.numeric_columns]

                for col in self.numeric_columns:
                    if col not in self.scalers:
                        self.scalers[col] = StandardScaler()

                    col_data = X_numeric[col].dropna().values.reshape(-1, 1)
                    if len(col_data) > 0:
                        self.scalers[col].partial_fit(col_data)

                    current_mean = X_numeric[col].mean()
                    if col in self.column_means:
                        # Update running average
                        n = len(X_numeric)
                        total = self.column_means[col] * n + current_mean * n
                        self.column_means[col] = total / (2 * n)
                    else:
                        self.column_means[col] = current_mean

                return self

            def transform(self, X):
                if isinstance(X, np.ndarray):
                    X = pd.DataFrame(X, columns=[f'col_{i}' for i in range(X.shape[1])])

                X_numeric = X[self.numeric_columns].copy()

                for col in self.numeric_columns:
                    col_data = X_numeric[col].values.reshape(-1, 1)
                    nan_mask = np.isnan(col_data)
                    col_data[nan_mask] = self.column_means[col]
                    X_numeric[col] = self.scalers[col].transform(col_data).flatten()
                    X_numeric.loc[nan_mask.flatten(), col] = np.nan

                X.loc[:,self.numeric_columns] = X_numeric
                return X

        import json
        import numpy as np

        def numpy_to_python(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: numpy_to_python(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [numpy_to_python(item) for item in obj]
            else:
                return obj

        #proficiency_scaler = NanPreservingScaler()

        #load working scaler (joblib)
        proficiency_scaler = joblib.load('./opt/ml/model/proficiency_scaler.joblib')

        #extract values
        scaler_dict = {
            'scalers': {k: {'mean_': v.mean_.tolist(), 'scale_': v.scale_.tolist(), 'var_': v.var_.tolist()} 
                        for k, v in proficiency_scaler.scalers.items()},
            'column_means': numpy_to_python(proficiency_scaler.column_means),
            'numeric_columns': proficiency_scaler.numeric_columns
        }

        # print(scaler_dict.keys())
        # print(scaler_dict)


        for k, v in scaler_dict['scalers'].items():
            scaler = StandardScaler()
            scaler.mean_ = np.array(v['mean_'])
            scaler.scale_ = np.array(v['scale_'])
            scaler.var_ = np.array(v['var_'])
            proficiency_scaler.scalers[k] = scaler

        proficiency_scaler.column_means = scaler_dict['column_means']
        proficiency_scaler.numeric_columns = scaler_dict['numeric_columns']


        #re export as 
        with open(new_scaler_location, 'w') as f:
            json.dump(scaler_dict, f, default=numpy_to_python)

        print('exporting to '+str(new_scaler_location))

        #change scaler location to new scaler
        print('old source_paths and new_names: ',source_paths['proficiency_scaler'], ' ', new_names['proficiency_scaler'])
        source_paths['proficiency_scaler'] = new_scaler_location
        new_names['proficiency_scaler'] = new_names['proficiency_scaler'].rsplit('.', 1)[0] + '.json'
        print('updated source_paths and new_names: ',source_paths['proficiency_scaler'], ' ', new_names['proficiency_scaler'])


In [3]:
#loads and converts percentile ranker from pkl to json

if source_paths['confidence_score_scaler'].rsplit('.', 1)[1] == 'pkl':
    ranker_location=source_paths['confidence_score_scaler']
    new_ranker_location = source_paths['confidence_score_scaler'].rsplit('.', 1)[0] + '.json'
    print('loading from '+str(ranker_location))
    # Load the confidence score scaler
    from scipy import stats
    class PercentileRankCalculator:
        def __init__(self, scores):
            self.scores = np.array(scores)
            self.ranks = stats.rankdata(self.scores, method='average')
            self.total_scores = len(self.scores)
        def get_percentile_rank(self, new_scores):
            new_scores = np.atleast_1d(new_scores)
            # Find where the new scores would be inserted
            insert_positions = np.searchsorted(self.scores, new_scores)
            # Calculate ranks
            new_ranks = np.where(insert_positions == 0, 1,
                        np.where(insert_positions == self.total_scores, self.total_scores,
                        np.where(new_scores == self.scores[np.maximum(insert_positions - 1, 0)],
                                 self.ranks[np.maximum(insert_positions - 1, 0)],
                                 insert_positions + 1)))
            # Calculate percentile ranks
            percentile_ranks = (new_ranks / (self.total_scores + 1)) * 100
            return percentile_ranks
        def save(self, filename):
            data = {
                'scores': self.scores.tolist(),
                'ranks': self.ranks.tolist(),
                'total_scores': self.total_scores
            }
            with open(filename, 'w') as f:
                json.dump(data, f)            
        @classmethod
        def load(cls, filename):
            with open(filename, 'r') as f:
                data = json.load(f)
            obj = cls([])  # Create an empty instance
            obj.scores = np.array(data['scores'])
            obj.ranks = np.array(data['ranks'])
            obj.total_scores = data['total_scores']
            return obj    
    # Load the confidence score scaler
    import json
    import pickle
    
    class CustomUnpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if module.startswith("ProficiencyModelTrainingPipeline"):
                return PercentileRankCalculator
            return globals().get(name, super().find_class(module, name))
            
    #load pkl
    with open(ranker_location, 'rb') as filename:
        confidence_score_scaler = CustomUnpickler(filename).load()
        
    #use class func to re-export json
    confidence_score_scaler.save(new_ranker_location)    
    print('exporting to '+str(new_ranker_location))
    #change scaler location to new scaler
    print('old source_paths and new_names: ',source_paths['confidence_score_scaler'], ' ', new_names['confidence_score_scaler'])
    source_paths['confidence_score_scaler'] = new_ranker_location
    new_names['confidence_score_scaler'] = new_names['confidence_score_scaler'].rsplit('.', 1)[0] + '.json'
    print('updated source_paths and new_names: ',source_paths['confidence_score_scaler'], ' ', new_names['confidence_score_scaler'])

    

loading from item_param_calc_50/confidence_model_01312025_noSkillExplode_v50Full_standard/percentile_rank_calculator.pkl
exporting to item_param_calc_50/confidence_model_01312025_noSkillExplode_v50Full_standard/percentile_rank_calculator.json
old source_paths and new_names:  item_param_calc_50/confidence_model_01312025_noSkillExplode_v50Full_standard/percentile_rank_calculator.pkl   confidence_score_scaler.pkl
updated source_paths and new_names:  item_param_calc_50/confidence_model_01312025_noSkillExplode_v50Full_standard/percentile_rank_calculator.json   confidence_score_scaler.json


In [4]:
#we'll move this to the run_pipeline tool eventually
#creates local test environment to test inference script
#also moves to docker build environment (Need to set up project first using Model Endpoint tools)

import os
import shutil

# Create the directory structure
# model_dir = os.path.join('opt', 'ml', 'model')
model_dir = '.'
os.makedirs(model_dir, exist_ok=True)



# Copy and rename files to the new directory
for key, source_path in source_paths.items():
    source_full_path = os.path.join(os.getcwd(), source_path)
    destination_path = os.path.join(model_dir, new_names[key])
    shutil.copy2(source_full_path, destination_path)
    print(f"Copied and renamed {key} to {destination_path}")

# Set up path location variables
proficiency_model_path = os.path.join(model_dir, new_names['proficiency_model'])
confidence_model_path = os.path.join(model_dir, new_names['confidence_model'])
if scalerUsed:
    proficiency_scaler_path = os.path.join(model_dir, new_names['proficiency_scaler'])
item_params_path = os.path.join(model_dir, new_names['item_params'])
confidence_score_scaler_path = os.path.join(model_dir, new_names['confidence_score_scaler'])

print("\nPath variables:")
print(f"Proficiency Model: {proficiency_model_path}")
print(f"Confidence Model: {confidence_model_path}")
if scalerUsed:
    print(f"Proficiency Scaler: {proficiency_scaler_path}")
print(f"Item Parameters: {item_params_path}")
print(f"Confidence Score Scaler: {confidence_score_scaler_path}")

# You can now use these path variables in your main script

#move to inference docker location
if moveArtifactsToInference:    
    # Copy and rename files to the new directory
    for key, source_path in source_paths.items():
        source_full_path = os.path.join(os.getcwd(), source_path)
        destination_path = os.path.join(artifact_destination_dir, new_names[key])
        shutil.copy2(source_full_path, destination_path)
        print(f"Copied and renamed {key}, {source_full_path} to {destination_path}")


Copied and renamed item_params to ./item_params.csv
Copied and renamed proficiency_model to ./proficiency_model.json
Copied and renamed confidence_model to ./confidence_model.json
Copied and renamed confidence_score_scaler to ./confidence_score_scaler.json

Path variables:
Proficiency Model: ./proficiency_model.json
Confidence Model: ./confidence_model.json
Item Parameters: ./item_params.csv
Confidence Score Scaler: ./confidence_score_scaler.json
Copied and renamed item_params, /home/ec2-user/SageMaker/[REDACTED]AppliedScience/Omar/WithinSkill_Math/item_param_calc_50/Math_ItemParameters_v5.csv to ../ModelImplementationWSDK/Math_student_proficiency_model/container/item_params.csv
Copied and renamed proficiency_model, /home/ec2-user/SageMaker/[REDACTED]AppliedScience/Omar/WithinSkill_Math/item_param_calc_50/proficiency_model_01312025_noSkillExplode_v50Full_standard/xgb_model.json to ../ModelImplementationWSDK/Math_student_proficiency_model/container/proficiency_model.json
Copied and rena

In [5]:
# !pip install xgboost

In [6]:
libraryVersionCheck=True

if libraryVersionCheck:
    import joblib
    print(joblib.__version__)

    import xgboost
    print(xgboost.__version__)

    import joblib
    #try saving with 'protocol 4'
    # joblib.dump(proficiency_scaler, '../ModelImplementationWSDK/ELA_student_proficiency_model_v0/container/proficiency_scaler.joblib', protocol=4)

    from sklearn.preprocessing import StandardScaler
    import sklearn
    print(sklearn.__version__)

    import joblib
    import sklearn
    import numpy as np
    import pandas as pd

    print(f"joblib version: {joblib.__version__}")
    print(f"sklearn version: {sklearn.__version__}")
    print(f"numpy version: {np.__version__}")
    print(f"pandas version: {pd.__version__}")

    !python --version

1.4.2
3.0.2
1.6.1
joblib version: 1.4.2
sklearn version: 1.6.1
numpy version: 1.26.4
pandas version: 1.5.3
Python 3.10.17


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [16]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.preprocessing import StandardScaler
import pickle
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
import xgboost as xgb
import joblib
import os
import json
from scipy import stats
from datetime import datetime

#Change parameters here
model_version = '5.0'+'5' #IPD version
HISTORY_LENGTH = 10
scalerUsed = False
debugInputs = True #exports feature engr'd inputs
batchSkillUsed = False #turns off skill aggregator (uses item model directly)
skillTransformerUsed = False #sigmoid scales predictions (closer to 0 & 1)

#Load all artifacts (5): 2 models, 1 scaler, 1 score transformer, IP file

# Load the proficiency model
proficiency_model = xgb.Booster()
proficiency_model.load_model('proficiency_model.json')

# Load the confidence model
confidence_model = xgb.Booster()
confidence_model.load_model('confidence_model.json')

# Load the proficiency scaler
class NanPreservingScaler:
    def __init__(self):
        self.scalers = {}
        self.column_means = {}
        self.numeric_columns = None

    def partial_fit(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f'col_{i}' for i in range(X.shape[1])])
        
        if self.numeric_columns is None:
            self.numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
        
        X_numeric = X[self.numeric_columns]
        
        for col in self.numeric_columns:
            if col not in self.scalers:
                self.scalers[col] = StandardScaler()
            
            col_data = X_numeric[col].dropna().values.reshape(-1, 1)
            if len(col_data) > 0:
                self.scalers[col].partial_fit(col_data)
            
            current_mean = X_numeric[col].mean()
            if col in self.column_means:
                # Update running average
                n = len(X_numeric)
                total = self.column_means[col] * n + current_mean * n
                self.column_means[col] = total / (2 * n)
            else:
                self.column_means[col] = current_mean
        
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f'col_{i}' for i in range(X.shape[1])])
        
        X_numeric = X[self.numeric_columns].copy()
        
        for col in self.numeric_columns:
            col_data = X_numeric[col].values.reshape(-1, 1)
            nan_mask = np.isnan(col_data)
            col_data[nan_mask] = self.column_means[col]
            X_numeric[col] = self.scalers[col].transform(col_data).flatten()
            X_numeric.loc[nan_mask.flatten(), col] = np.nan
        
        X.loc[:,self.numeric_columns] = X_numeric
        return X

#load working scaler (joblib)
#proficiency_scaler = joblib.load('./opt/ml/model/proficiency_scaler.joblib')

if scalerUsed:
    with open('proficiency_scaler.json', 'r') as f:
        scaler_dict = json.load(f)

    proficiency_scaler = NanPreservingScaler()
    proficiency_scaler.scalers = {}
    for k, v in scaler_dict['scalers'].items():
        scaler = StandardScaler()
        scaler.mean_ = np.array(v['mean_'])
        scaler.scale_ = np.array(v['scale_'])
        scaler.var_ = np.array(v['var_'])
        proficiency_scaler.scalers[k] = scaler

    proficiency_scaler.column_means = scaler_dict['column_means']
    proficiency_scaler.numeric_columns = scaler_dict['numeric_columns']
else:
    proficiency_scaler=None


def apply_sigmoid(predictions, params):
    a, b = params
    return 1 / (1 + np.exp(-a * (predictions - b)))  # b is the centering point

if skillTransformerUsed:
    with open('sigmoid_params.json', 'r') as f:
        params = json.load(f)
        try:
            sigmoid_params = [params['a'], params['b']] 
        except:
            sigmoid_params = [params['a'], params['b'], params['future_window']]
            sigmoid_params = sigmoid_params[:1]
            
else:
    sigmoid_params = None

# Load the confidence score scaler
class PercentileRankCalculator:
    def __init__(self, scores):
        self.scores = np.array(scores)
        self.ranks = stats.rankdata(self.scores, method='average')
        self.total_scores = len(self.scores)
    
    def get_percentile_rank(self, new_scores):
        new_scores = np.atleast_1d(new_scores)
        
        # Find where the new scores would be inserted
        insert_positions = np.searchsorted(self.scores, new_scores)
        
        # Calculate ranks
        new_ranks = np.where(insert_positions == 0, 1,
                    np.where(insert_positions == self.total_scores, self.total_scores,
                    np.where(new_scores == self.scores[np.maximum(insert_positions - 1, 0)],
                             self.ranks[np.maximum(insert_positions - 1, 0)],
                             insert_positions + 1)))
        
        # Calculate percentile ranks
        percentile_ranks = (new_ranks / (self.total_scores + 1)) * 100
        
        return percentile_ranks

    #def save(self, filename):
    #    with open(filename, 'wb') as f:
    #        pickle.dump(self, f)
            
    def save(self, filename):
        data = {
            'scores': self.scores.tolist(),
            'ranks': self.ranks.tolist(),
            'total_scores': self.total_scores
        }
        with open(filename, 'w') as f:
            json.dump(data, f)            
    
#     @classmethod
#     def load(cls, filename):
#         with open(filename, 'rb') as f:
#             return pickle.load(f)
    @classmethod
    def load(cls, filename):
        with open(filename, 'r') as f:
            data = json.load(f)
        obj = cls([])  # Create an empty instance
        obj.scores = np.array(data['scores'])
        obj.ranks = np.array(data['ranks'])
        obj.total_scores = data['total_scores']
        return obj    

    
# Load the confidence score scaler
with open('confidence_score_scaler.json', 'r') as f:
    scaler_data = json.load(f)
confidence_score_scaler = PercentileRankCalculator([])
confidence_score_scaler.scores = np.array(scaler_data['scores'])
confidence_score_scaler.ranks = np.array(scaler_data['ranks'])
confidence_score_scaler.total_scores = scaler_data['total_scores']    
    
    
# with open('./opt/ml/model/confidence_score_scaler.pkl', 'rb') as f:
#     confidence_score_scaler = pickle.load(f)
# confidence_score_scaler.save('./opt/ml/model/confidence_score_scaler.json')    



# Load the item parameters
item_params = pd.read_csv('item_params.csv')
#basically gets rid of 'version', other stored metadata(?)
item_params = item_params[['question_id', 'skill_id', 'discriminability', 'difficulty', 
                               'guessing','inattention', 'discriminability_error', 
                               'difficulty_error', 'guessing_error', 'inattention_error', 
                               'auc_roc', 'optimal_threshold', 'tpr', 'tnr', 'skill_optimal_threshold',
                               'student_mean_accuracy', 'sample_size']]

#create a copy where we pre-transform sample size (for horizontal batch inference)
transformed_loaded_ipd = item_params.copy()
transformed_loaded_ipd['LOG_sample_size'] = np.log(transformed_loaded_ipd['sample_size'])
transformed_loaded_ipd = transformed_loaded_ipd.drop('sample_size', axis=1)

# Make list of all valid skill_ids (for horiz batch inference)
valid_skill_ids = set(transformed_loaded_ipd['skill_id'].unique())
    
print("All models, artifacts, and IP data loaded successfully.")

def validate_input(data):
    """Validates and cleans input data."""
    required_fields = ['skillId', 'questionId', 'eventTime', 'questionIdsHistory', 
                      'correctnessHistory', 'durationSecondsHistory', 'eventTimesHistory']
    
    # Check for missing required fields
    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")
        
    # Ensure all histories are lists
    for field in ['questionIdsHistory', 'correctnessHistory', 'durationSecondsHistory', 'eventTimesHistory']:
        if not isinstance(data[field], list):
            data[field] = [data[field]] if data[field] is not None else []

    return data
    
def process_input_original(data):
    """Process input data without lag shifting for skill predictions"""
    try:
        data = validate_input(data)
        
        features = {
            'SKILL': [data['skillId']],
            'QUESTIONID': [data['questionId']],
            'OCCURRED_AT': [pd.to_datetime(data['eventTime'])]
        }
        
        # Original lag structure (1 to HISTORY_LENGTH)
        for i in range(1, HISTORY_LENGTH + 1):
            features[f'QUESTIONID_LAG_{i}'] = [
                data['questionIdsHistory'][i-1] if i-1 < len(data['questionIdsHistory']) else np.nan
            ]
            features[f'CORRECTNESS_LAG_{i}'] = [
                float(data['correctnessHistory'][i-1]) 
                if i-1 < len(data['correctnessHistory']) and data['correctnessHistory'][i-1] is not None 
                else np.nan
            ]
            features[f'DURATIONSECONDS_LAG_{i}'] = [
                float(data['durationSecondsHistory'][i-1])
                if i-1 < len(data['durationSecondsHistory']) and data['durationSecondsHistory'][i-1] is not None
                else np.nan
            ]
            features[f'OCCURREDAT_LAG_{i}'] = [
                pd.to_datetime(data['eventTimesHistory'][i-1])
                if i-1 < len(data['eventTimesHistory']) and data['eventTimesHistory'][i-1]
                else pd.NaT
            ]
        
        return pd.DataFrame(features)
    except Exception as e:
        raise ValueError(f"Error processing input: {str(e)}")

def process_input_lagged(data):
    """Process input data with lag shifting for item predictions"""
    #API is inexplicably, despite specifications 
    #https://illuminate.atlassian.net/wiki/spaces/~42953498/pages/17608311471/ELA+Student+Proficiency+Endpoint+Inputs+and+Outputs
    #double submitting the last item as history AND current
    #this will deliberately ignore the first history item ('lagged' model metrics)
    #but only for item, not the live skill prediction

    try:
        data = validate_input(data)
        
        features = {
            'SKILL': [np.nan if not data['skillId'] else data['skillId']],
            'QUESTIONID': [np.nan if not data['questionIdsHistory'] else data['questionIdsHistory'][0]],
            'OCCURRED_AT': [np.nan if not data['eventTimesHistory'] else pd.to_datetime(data['eventTimesHistory'][0])]
        }
        # Shifted lag structure (2 to HISTORY_LENGTH+1)
        # start w indx 1 instead of 0: [i-1], 2,HL+2: 1, HL+1
        for i in range(2, HISTORY_LENGTH + 2):
            features[f'QUESTIONID_LAG_{i-1}'] = [
                data['questionIdsHistory'][i-1] if i-1 < len(data['questionIdsHistory']) else np.nan
            ]
            features[f'CORRECTNESS_LAG_{i-1}'] = [
                float(data['correctnessHistory'][i-1]) 
                if i-1 < len(data['correctnessHistory']) and data['correctnessHistory'][i-1] is not None 
                else np.nan
            ]
            features[f'DURATIONSECONDS_LAG_{i-1}'] = [
                float(data['durationSecondsHistory'][i-1])
                if i-1 < len(data['durationSecondsHistory']) and data['durationSecondsHistory'][i-1] is not None
                else np.nan
            ]
            features[f'OCCURREDAT_LAG_{i-1}'] = [
                pd.to_datetime(data['eventTimesHistory'][i-1])
                if i-1 < len(data['eventTimesHistory']) and data['eventTimesHistory'][i-1]
                else pd.NaT
            ]
        
        return pd.DataFrame(features)
    except Exception as e:
        raise ValueError(f"Error processing input: {str(e)}")


def process_inference_data(df, item_params):
    # Load ItemParameters already loaded (item_params)
        
    # Step 0: CORRECTNESS
    correctness_columns = ['CORRECTNESS'] + [f'CORRECTNESS_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    for col in correctness_columns:
        if col in df.columns:
            #df[col] = pd.to_numeric(df[col], errors='coerce') / 100
            #API is inexplicably, despite specifications 
            #https://illuminate.atlassian.net/wiki/spaces/~42953498/pages/17608311471/ELA+Student+Proficiency+Endpoint+Inputs+and+Outputs
            #dividiing by 100 before submission
            df[col] = pd.to_numeric(df[col], errors='coerce')
            #I have a lot of unit tests, test data, etc so this is a unoptimal patch for now for the API
            # Normalize only if the column is not empty and max > 1
            if len(df[col].dropna()) > 0 and df[col].max() > 1:
                df[col] = df[col] / 100

    # Step 1: Calculate time differences
    occurred_columns = ['OCCURRED_AT'] + [f'OCCURREDAT_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]

    #timezone formatting issues
    for col in occurred_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
            if df[col].dt.tz is None:
                df[col] = pd.DatetimeIndex(df[col]).tz_localize('UTC')
            else:
                df[col] = df[col].dt.tz_convert('UTC')
   
    for i in range(1, len(occurred_columns)):
        if occurred_columns[i-1] in df.columns and occurred_columns[i] in df.columns:
            diff_col = f'OCCURREDAT_DIFF_{i}'
            #df[diff_col] = (pd.to_datetime(df[occurred_columns[i-1]]) - pd.to_datetime(df[occurred_columns[i]])).dt.total_seconds() / 3600
            #df[diff_col] = ((df[occurred_columns[i-1]] - df[occurred_columns[i]]).dt.total_seconds() / 3600).round(6)
            # Make the operations more explicit
            time_diff_seconds = (df[occurred_columns[i-1]] - df[occurred_columns[i]]).dt.total_seconds()
            time_diff_hours = time_diff_seconds / 3600
            df[diff_col] = np.round(time_diff_hours, decimals=3)
        
    # Drop OCCURRED_AT columns
    df = df.drop(columns=[col for col in occurred_columns if col in df.columns])

    # Step 3a: Minimum cap OCCURREDAT_DIFF to 1/3600
    diff_columns = [col for col in df.columns if col.startswith('OCCURREDAT_DIFF')]
    for col in diff_columns:
        df[col] = df[col].clip(lower=1/3600)

    # Step 3b and 3c: Cap DURATIONSECONDS
    duration_columns = [f'DURATIONSECONDS_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    for col in duration_columns:
        if col in df.columns:
            df[col] = df[col].clip(lower=1, upper=300)

    # Step 4: Log transform
    for col in diff_columns + duration_columns:
        if col in df.columns:
            df[f'LOG_{col}'] = np.log(df[col])

    # Step 5: LEFT JOIN with ItemParameters
    for i in range(HISTORY_LENGTH+1): #0 is current, 1-10 lag 
        suffix = f'_LAG_{i}' if i > 0 else ''
        question_col = 'QUESTIONID' if i == 0 else f'QUESTIONID_LAG_{i}'
        if question_col in df.columns:
            df[question_col] = df[question_col].fillna('nan').astype(str)
            temp_df = item_params.copy()
            temp_df.columns = [f'{col}{suffix}' for col in temp_df.columns]
            df = df.merge(temp_df, left_on=[question_col, 'SKILL'], 
                          right_on=[f'question_id{suffix}', f'skill_id{suffix}'], how='left')
            
            # Drop the join columns
            df = df.drop(columns=[f'question_id{suffix}', f'skill_id{suffix}'])
            
    # Step 5b: EXTRA STEP FOR INFERENCE: DROP questions
    qid_columns = [f'QUESTIONID_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    df = df.drop(columns=[col for col in qid_columns if col in df.columns])

    # Step 6: Log transform sample_size and remove original
    sample_size_columns = ['sample_size'] + [f'sample_size_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    for col in sample_size_columns:
        if col in df.columns:
            df[f'LOG_{col}'] = np.log(df[col])
            df = df.drop(columns=[col])

    # Step 7: Add spread features
    for feature in ['difficulty', 'optimal_threshold', 'student_mean_accuracy', 'tnr', 'tpr']:
        cols = [f'{feature}_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
        if all(col in df.columns for col in cols):
            df[f'{feature}_spread'] = df[cols].max(axis=1) - df[cols].min(axis=1)

    # Step 8: Add mean of auc_roc and discriminability and error
    auc_cols = [f'auc_roc_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    disc_cols = [f'discriminability_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    if all(col in df.columns for col in auc_cols):
        df['mean_auc_roc'] = df[auc_cols].mean(axis=1)
    if all(col in df.columns for col in disc_cols):
        df['mean_discriminability'] = df[disc_cols].mean(axis=1)

    error_features = ['discriminability_error', 'difficulty_error', 'guessing_error', 'inattention_error']
    for feature in error_features:
        feature_cols = [f'{feature}_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
        if all(col in df.columns for col in feature_cols):
            df[f'mean_{feature}'] = df[feature_cols].mean(axis=1)

    # Step 9: drop redundant skill OTs
    skill_optimal_threshold_columns = [f'skill_optimal_threshold_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    df = df.drop(columns=[col for col in skill_optimal_threshold_columns if col in df.columns])

    # Step 10: add num responses
    correctness_columns = [f'CORRECTNESS_LAG_{i}' for i in range(1, HISTORY_LENGTH+1)]
    df['num_responses'] = df[[col for col in correctness_columns if col in df.columns]].notna().sum(axis=1).astype(float)

    return df
    

def run_model_inference(df, proficiency_model, confidence_model, proficiency_scaler, confidence_score_scaler):
    """
    Run model inference on a single item/skill combination.
    """
    
    if scalerUsed:
        # Scale data with proficiency_scaler
        scaled_data = proficiency_scaler.transform(df)
    else:
        scaled_data = df[:]
    
    # Run inference using proficiency_model
    dmatrix = xgb.DMatrix(scaled_data)
    if skillTransformerUsed:
        item_prediction = apply_sigmoid(proficiency_model.predict(dmatrix),sigmoid_params)
    else:
        item_prediction = np.clip(proficiency_model.predict(dmatrix),0,1)
    
    # Run inference using confidence_model
    confidence_input = scaled_data.copy()
    confidence_input['proficiency_model_prediction'] = item_prediction
    confidence_dmatrix = xgb.DMatrix(confidence_input)
    item_prediction_error = np.clip(confidence_model.predict(confidence_dmatrix),0,1)
    
    # Scale the confidence score
    item_prediction_confidence = confidence_score_scaler.get_percentile_rank(1- item_prediction_error)

    return item_prediction, item_prediction_error, item_prediction_confidence

def batch_predict_all_questions_for_skills_w_conf(proficiency_model, confidence_model, 
                                                  transformed_loaded_ipd, X_input, skill_ids, 
                                                  feature_names, proficiency_scaler):
    unique_skill_ids = np.unique(skill_ids)
    all_predictions = []
    all_question_ids = []
    
    #we use transformed loaded ipd (which just pre-log txs sample size)
    #and ignores version
    transformed_ipd_features = ['discriminability', 'difficulty', 'guessing', 'inattention',
                    'discriminability_error', 'difficulty_error', 'guessing_error',
                    'inattention_error', 'auc_roc', 'optimal_threshold', 'tpr', 'tnr',
                    'student_mean_accuracy', 'LOG_sample_size']
    
    
        
    # Pre-process IPD data/condense to what we need
    transformed_ipd_dict = {skill: transformed_loaded_ipd[transformed_loaded_ipd['skill_id'] == skill].groupby('question_id').last() for skill in unique_skill_ids}
    
    prediction_counts = []  # To keep track of number of predictions per input row
    valid_indices = [] # to track bad submissions with no valid skill_id
    
    for i, skill_id in enumerate(skill_ids):
        if skill_id not in valid_skill_ids:
            prediction_counts.append(0)
            continue

        valid_indices.append(i)
        X_skill = X_input.iloc[i:i+1]  # Get single row
        
        question_ids = transformed_ipd_dict[skill_id].index
        n_questions = len(question_ids)
        
        if n_questions == 0:
            prediction_counts.append(0)
            continue
            
        # Create an array to hold all question variations
        all_questions = np.repeat(X_skill.values, n_questions, axis=0)
        
        # Update the question-specific columns for each question
        for col in transformed_ipd_features:
            all_questions[:, feature_names.index(col)] = transformed_ipd_dict[skill_id][col].values
        
        all_predictions.append(all_questions)
        all_question_ids.extend(question_ids)
        prediction_counts.append(n_questions)
    
    #possibly unnecessary check -- return empties if function got called w no valid skill_id
    if not all_predictions:
        return (np.array([]), np.array([]), np.array([]), [], prediction_counts, valid_indices)

    # Combine all predictions
    all_predictions = np.vstack(all_predictions)
    
    #convert to df
    all_predictions = pd.DataFrame(all_predictions, columns=feature_names)
    
    if scalerUsed:
        # Scale the data
        X_all_questions_scaled = proficiency_scaler.transform(all_predictions)
    else:
        X_all_questions_scaled = all_predictions.copy()
        

    # Make predictions for all questions at once
    dmatrix = xgb.DMatrix(X_all_questions_scaled)
    
    #clip or sigmoid the predictions
    if skillTransformerUsed:
        predictions = apply_sigmoid(proficiency_model.predict(dmatrix),sigmoid_params)
    else:
        predictions = np.clip(proficiency_model.predict(dmatrix),0,1)

    
    # Prepare input for confidence model
    #confidence_input = np.column_stack((X_all_questions_scaled, predictions))
    X_all_questions_scaled['proficiency_model_prediction']=predictions
    
    # Make confidence predictions
    confidence_dmatrix = xgb.DMatrix(X_all_questions_scaled)
    confidence_predicted_error = np.clip(confidence_model.predict(confidence_dmatrix),0,1)
    
    confidence_scores = confidence_score_scaler.get_percentile_rank(1 - confidence_predicted_error)

    
    return predictions, confidence_predicted_error, confidence_scores, all_question_ids, prediction_counts, valid_indices

def run_inference(data):
    """
    Run comprehensive inference on input data.
    """
    # Process the input data two ways
    if isinstance(data, dict):  # Single inference
        input_df_original = process_input_original(data)
        input_df_lagged = process_input_lagged(data)
    elif isinstance(data, list):  # Batch inference
        input_df_original = pd.concat([process_input_original(item) for item in data], ignore_index=True)
        input_df_lagged = pd.concat([process_input_lagged(item) for item in data], ignore_index=True)
    else:
        raise ValueError("Input data must be a dictionary for single inference or a list of dictionaries for batch inference")
            
    # Run lagged item inference (for metrics tracking)
    # Process inference data
    processed_df_lagged = process_inference_data(input_df_lagged, item_params)    
    processed_df_lagged = processed_df_lagged.drop(['QUESTIONID', 'SKILL'], axis=1)
    # Reorder features
    correct_feature_order = proficiency_model.feature_names
    reordered_df_lagged = processed_df_lagged[correct_feature_order]
    
    
    lagged_item_predictions, lagged_item_prediction_errors, lagged_item_prediction_confidences = run_model_inference(
        reordered_df_lagged, proficiency_model, confidence_model, proficiency_scaler, confidence_score_scaler
    )
        
    # Run batch prediction for skills (using original)
    if batchSkillUsed:
        
        # Process inference data 
        processed_df_original = process_inference_data(input_df_original, item_params)
        # Extract required information for batch prediction (using original)
        skill_ids = processed_df_original['SKILL'].tolist()
        processed_df_original = processed_df_original.drop(['QUESTIONID', 'SKILL'], axis=1)
        # Reorder features 
        reordered_df_original = processed_df_original[correct_feature_order]
        feature_names = reordered_df_original.columns.tolist()

        
        predictions, confidence_predicted_error, confidence_scores, all_question_ids, prediction_counts, valid_indices = batch_predict_all_questions_for_skills_w_conf(
            proficiency_model, confidence_model, transformed_loaded_ipd, reordered_df_original, 
            skill_ids, feature_names, proficiency_scaler
        )

        # Process skill predictions as before
        skill_predictions_means = np.full(len(skill_ids), np.nan)
        skill_uncertainity_means = np.full(len(skill_ids), np.nan)
        skill_confidence_score_means = np.full(len(skill_ids), np.nan)

        if len(predictions) > 0:
            valid_predictions = np.split(predictions, np.cumsum(prediction_counts)[:-1])
            valid_uncertainties = np.split(confidence_predicted_error, np.cumsum(prediction_counts)[:-1])
            valid_confidences = np.split(confidence_scores, np.cumsum(prediction_counts)[:-1])

            for i, idx in enumerate(valid_indices):
                if prediction_counts[idx] > 0:
                    skill_predictions_means[idx] = np.mean(valid_predictions[i])
                    skill_uncertainity_means[idx] = np.mean(valid_uncertainties[i])
                    skill_confidence_score_means[idx] = np.mean(valid_confidences[i])
    #else just do 'item agnostic' prediction
    else:        
        # Process inference data WITHOUT next-item information (API IS INCORRECT HERE TOO)
        #DROP Erroneous questionid first, before processing
        processed_df_original = input_df_original.drop(['QUESTIONID'], axis=1)
        processed_df_original = process_inference_data(input_df_original, item_params)
        processed_df_original = processed_df_original.drop(['SKILL'], axis=1)
        # Extract required information for batch prediction (using original)
        # Reorder features 
        reordered_df_original = processed_df_original[correct_feature_order]
        feature_names = reordered_df_original.columns.tolist()


        skill_predictions_means, skill_uncertainity_means, skill_confidence_score_means = run_model_inference(
            reordered_df_original, proficiency_model, confidence_model, proficiency_scaler, confidence_score_scaler
        )


    output_df = pd.DataFrame({
        'item_prediction': lagged_item_predictions,
        'item_prediction_error': lagged_item_prediction_errors,
        'item_prediction_confidence': lagged_item_prediction_confidences,
        'skill_prediction': skill_predictions_means,
        'skill_prediction_error': skill_uncertainity_means,
        'skill_prediction_confidence': skill_confidence_score_means
    })
    
    output_df = output_df.replace({np.nan: None})
    
    if debugInputs:
        return output_df, reordered_df_lagged
    else:
        return output_df

All models, artifacts, and IP data loaded successfully.


In [17]:
#RUN ABOVE CELL before this cell (if you're running cells out of order)
#Code to copy above inference code and dump it into an inference file
#Code below wraps this with a Flask script

def export_cell_with_flask_wrapper(cell_index, output_path):
    """
    Export a single cell from the current notebook and wrap it with Flask boilerplate code
    
    Parameters:
    cell_index (int): Index of the cell to export (0-based)
    output_path (str): Path for the output .py file
    """
    from IPython import get_ipython
    
    # Get the current notebook's cells
    ipython = get_ipython()
    cells = ipython.ev('In')  # Gets all input cells
    
    if cell_index >= len(cells):
        raise IndexError(f"Cell index {cell_index} is out of range. Notebook has {len(cells)} cells.")
        
    # Get the source code from the specified cell
    cell_code = cells[cell_index]
    
    # Flask header code
    flask_header = '''import flask
from flask import Flask, request, jsonify
# Initialize the Flask app
app = Flask(__name__)
#
#Inference Wrapper starts here
'''

    # Flask footer code
    flask_footer = '''#
#Inference Wrapper ends here
#
@app.route('/ping', methods=['GET'])
def ping():
    """
    Determine if the container is working and healthy.
    """
    return flask.Response(response='\\n', status=200, mimetype='application/json')


@app.route('/invocations', methods=['POST'])
def transformation():
    """
    Do an inference on a single batch of data.
    """
    debug_info = {
        'received_content_type': flask.request.content_type,
        'received_data_length': len(flask.request.get_data()),
        'received_raw_data': flask.request.get_data().decode('utf-8', errors='replace')
    }
    
    if flask.request.content_type != 'application/json':
        error_response = {
            'error': 'This predictor only supports JSON data',
            'debug_info': debug_info
        }
        return flask.Response(
            response=json.dumps(error_response),
            status=415,
            mimetype='application/json'
        )
    
    try:
        data = flask.request.get_json()
        debug_info['parsed_json_length'] = len(str(data))
        debug_info['parsed_json'] = data
        
        if debugInputs: #debug
            results, feature_engineered_input = run_inference(data)
            #force only one row 
            #debug_info['feature_engineered_input'] = feature_engineered_input.iloc[0].to_dict()
            debug_info['feature_engineered_input'] = feature_engineered_input.iloc[0].replace({np.nan: None}).to_dict()
        else:
            results = run_inference(data)
        
        results['model_version'] = model_version
        
        # Add debug info to successful response while keeping original structure
        response_data = {
            'prediction': results.to_dict(orient='list'),
            'debug_info': debug_info
        }
        
        return jsonify(response_data)
        
    except json.JSONDecodeError as e:
        error_response = {
            'error': 'Failed to parse JSON data',
            'error_details': {
                'error_type': 'JSONDecodeError',
                'error_message': str(e),
                'error_position': f'line {e.lineno}, column {e.colno}',
                'error_doc': e.doc
            },
            'debug_info': debug_info
        }
        return flask.Response(
            response=json.dumps(error_response),
            status=400,
            mimetype='application/json'
        )
    except Exception as e:
        error_response = {
            'error': 'An unexpected error occurred while processing the request',
            'error_details': {
                'error_type': type(e).__name__,
                'error_message': str(e)
            },
            'debug_info': debug_info
        }
        return flask.Response(
            response=json.dumps(error_response),
            status=500,
            mimetype='application/json'
        )

@app.errorhandler(404)
def not_found_error(error):
    return flask.Response(
        response=json.dumps({
            'error': 'Resource not found',
            'error_details': {
                'error_type': '404',
                'error_message': 'The requested URL was not found on the server.'
            }
        }),
        status=404,
        mimetype='application/json'
    )


if __name__ == "__main__":
    app.run(host='0.0.0.0', port=8080)
'''
    
    # Write to file
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(flask_header)
        f.write('\n')
        f.write(cell_code)
        f.write('\n')
        f.write(flask_footer)
        
    print(f"Cell {cell_index} exported with Flask wrapper to {output_path}")
    

if writeOutInferenceFile:
    # Export prev cell with Flask wrapper
    current_cell = len(In) - 2
    export_cell_with_flask_wrapper(current_cell, artifact_destination_dir + 'inference.py')    
    print('exported inference_test.py to '+artifact_destination_dir+ ' with flask wrapper and inference code.')

Cell 16 exported with Flask wrapper to ../ModelImplementationWSDK/Math_student_proficiency_model/container/inference.py
exported inference_test.py to ../ModelImplementationWSDK/Math_student_proficiency_model/container/ with flask wrapper and inference code.


In [18]:
# Test code
#generate random test samples
NUM_HISTORY=10
print('numhistory', NUM_HISTORY)

import json
from datetime import datetime, timedelta
import random
import string

#make list of real and fake skillids
valid_skills = list(valid_skill_ids)[:5]
fake_skills = []
for _ in range(5):
    fake_id = 'FAKE_' + ''.join(random.choices(string.ascii_uppercase + string.digits, k=5))
    fake_skills.append(fake_id)
all_skills = valid_skills + fake_skills

def generate_sample_input(num_history=NUM_HISTORY):
    current_time = datetime.now()
    
    return {
        'skillId': f'{random.choice(all_skills)}',
        'questionId': f'question_{random.randint(1, 1000)}',
        'eventTime': current_time.isoformat(),
        'questionIdsHistory': [f'question_{random.randint(1, 1000)}' for _ in range(num_history)],
        'correctnessHistory': [random.choice([0, 100]) for _ in range(num_history)],
        'durationSecondsHistory': [random.randint(10, 300) for _ in range(num_history)],
        'eventTimesHistory': [(current_time - timedelta(minutes=i*10)).isoformat() for i in range(num_history)]
    }

# Generate a single input
single_input = generate_sample_input()

print("Single Input Sample:")
print(json.dumps(single_input, indent=2))

# Generate multiple inputs
num_samples = 5
multiple_inputs = [generate_sample_input() for _ in range(num_samples)]

print("\nMultiple Inputs Sample (first two entries):")
print(json.dumps(multiple_inputs[:2], indent=2))

# If you need these as JSON strings for API testing:
single_input_json = json.dumps(single_input)
multiple_inputs_json = json.dumps(multiple_inputs)

print("\nSingle Input JSON string:")
print(single_input_json)

print("\nMultiple Inputs JSON string (first 100 characters):")
print(multiple_inputs_json[:100] + "...")

# Function to verify the structure of the input
def verify_input_structure(input_data):
    required_keys = ['skillId', 'questionId', 'eventTime', 'questionIdsHistory', 
                     'correctnessHistory', 'durationSecondsHistory', 'eventTimesHistory']
    
    if isinstance(input_data, dict):
        # Single input
        missing_keys = [key for key in required_keys if key not in input_data]
        if missing_keys:
            print(f"Warning: Missing keys in input: {missing_keys}")
        else:
            print("Input structure is correct.")
    elif isinstance(input_data, list):
        # Multiple inputs
        for i, item in enumerate(input_data):
            missing_keys = [key for key in required_keys if key not in item]
            if missing_keys:
                print(f"Warning: Missing keys in input {i}: {missing_keys}")
            else:
                print(f"Input {i} structure is correct.")
    else:
        print("Invalid input type. Expected dict or list of dicts.")

print("\nVerifying Single Input Structure:")
verify_input_structure(single_input)

print("\nVerifying Multiple Inputs Structure:")
verify_input_structure(multiple_inputs)

numhistory 10
Single Input Sample:
{
  "skillId": "FAKE_U2TYH",
  "questionId": "question_651",
  "eventTime": "2025-05-27T21:24:38.579718",
  "questionIdsHistory": [
    "question_47",
    "question_648",
    "question_972",
    "question_46",
    "question_250",
    "question_428",
    "question_502",
    "question_915",
    "question_785",
    "question_600"
  ],
  "correctnessHistory": [
    100,
    0,
    100,
    0,
    0,
    100,
    0,
    100,
    0,
    0
  ],
  "durationSecondsHistory": [
    38,
    282,
    266,
    23,
    187,
    120,
    157,
    179,
    220,
    211
  ],
  "eventTimesHistory": [
    "2025-05-27T21:24:38.579718",
    "2025-05-27T21:14:38.579718",
    "2025-05-27T21:04:38.579718",
    "2025-05-27T20:54:38.579718",
    "2025-05-27T20:44:38.579718",
    "2025-05-27T20:34:38.579718",
    "2025-05-27T20:24:38.579718",
    "2025-05-27T20:14:38.579718",
    "2025-05-27T20:04:38.579718",
    "2025-05-27T19:54:38.579718"
  ]
}

Multiple Inputs Sample (first 

In [19]:
single_input

{'skillId': 'FAKE_U2TYH',
 'questionId': 'question_651',
 'eventTime': '2025-05-27T21:24:38.579718',
 'questionIdsHistory': ['question_47',
  'question_648',
  'question_972',
  'question_46',
  'question_250',
  'question_428',
  'question_502',
  'question_915',
  'question_785',
  'question_600'],
 'correctnessHistory': [100, 0, 100, 0, 0, 100, 0, 100, 0, 0],
 'durationSecondsHistory': [38, 282, 266, 23, 187, 120, 157, 179, 220, 211],
 'eventTimesHistory': ['2025-05-27T21:24:38.579718',
  '2025-05-27T21:14:38.579718',
  '2025-05-27T21:04:38.579718',
  '2025-05-27T20:54:38.579718',
  '2025-05-27T20:44:38.579718',
  '2025-05-27T20:34:38.579718',
  '2025-05-27T20:24:38.579718',
  '2025-05-27T20:14:38.579718',
  '2025-05-27T20:04:38.579718',
  '2025-05-27T19:54:38.579718']}

In [20]:
#check output of single iference test
if debugInputs:
    results, processed_input = run_inference(single_input)
else:
    results = run_inference(single_input)
results

,item_prediction,item_prediction_error,item_prediction_confidence,skill_prediction,skill_prediction_error,skill_prediction_confidence
0,0.184189,0.46866,6.252167,0.567032,0.561268,0.000806


In [21]:
# Comprehensive solution to show everything
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # This prevents truncation of column contents

if debugInputs:
     display(processed_input)

,CORRECTNESS_LAG_1,CORRECTNESS_LAG_2,CORRECTNESS_LAG_3,CORRECTNESS_LAG_4,CORRECTNESS_LAG_5,CORRECTNESS_LAG_6,CORRECTNESS_LAG_7,CORRECTNESS_LAG_8,CORRECTNESS_LAG_9,CORRECTNESS_LAG_10,DURATIONSECONDS_LAG_1,DURATIONSECONDS_LAG_2,DURATIONSECONDS_LAG_3,DURATIONSECONDS_LAG_4,DURATIONSECONDS_LAG_5,DURATIONSECONDS_LAG_6,DURATIONSECONDS_LAG_7,DURATIONSECONDS_LAG_8,DURATIONSECONDS_LAG_9,DURATIONSECONDS_LAG_10,OCCURREDAT_DIFF_1,OCCURREDAT_DIFF_2,OCCURREDAT_DIFF_3,OCCURREDAT_DIFF_4,OCCURREDAT_DIFF_5,OCCURREDAT_DIFF_6,OCCURREDAT_DIFF_7,OCCURREDAT_DIFF_8,OCCURREDAT_DIFF_9,OCCURREDAT_DIFF_10,LOG_OCCURREDAT_DIFF_1,LOG_OCCURREDAT_DIFF_2,LOG_OCCURREDAT_DIFF_3,LOG_OCCURREDAT_DIFF_4,LOG_OCCURREDAT_DIFF_5,LOG_OCCURREDAT_DIFF_6,LOG_OCCURREDAT_DIFF_7,LOG_OCCURREDAT_DIFF_8,LOG_OCCURREDAT_DIFF_9,LOG_OCCURREDAT_DIFF_10,LOG_DURATIONSECONDS_LAG_1,LOG_DURATIONSECONDS_LAG_2,LOG_DURATIONSECONDS_LAG_3,LOG_DURATIONSECONDS_LAG_4,LOG_DURATIONSECONDS_LAG_5,LOG_DURATIONSECONDS_LAG_6,LOG_DURATIONSECONDS_LAG_7,LOG_DURATIONSECONDS_LAG_8,LOG_DURATIONSECONDS_LAG_9,LOG_DURATIONSECONDS_LAG_10,discriminability,difficulty,guessing,inattention,discriminability_error,difficulty_error,guessing_error,inattention_error,auc_roc,optimal_threshold,tpr,tnr,skill_optimal_threshold,student_mean_accuracy,discriminability_LAG_1,difficulty_LAG_1,guessing_LAG_1,inattention_LAG_1,discriminability_error_LAG_1,difficulty_error_LAG_1,guessing_error_LAG_1,inattention_error_LAG_1,auc_roc_LAG_1,optimal_threshold_LAG_1,tpr_LAG_1,tnr_LAG_1,student_mean_accuracy_LAG_1,discriminability_LAG_2,difficulty_LAG_2,guessing_LAG_2,inattention_LAG_2,discriminability_error_LAG_2,difficulty_error_LAG_2,guessing_error_LAG_2,inattention_error_LAG_2,auc_roc_LAG_2,optimal_threshold_LAG_2,tpr_LAG_2,tnr_LAG_2,student_mean_accuracy_LAG_2,discriminability_LAG_3,difficulty_LAG_3,guessing_LAG_3,inattention_LAG_3,discriminability_error_LAG_3,difficulty_error_LAG_3,guessing_error_LAG_3,inattention_error_LAG_3,auc_roc_LAG_3,optimal_threshold_LAG_3,tpr_LAG_3,tnr_LAG_3,student_mean_accuracy_LAG_3,discriminability_LAG_4,difficulty_LAG_4,guessing_LAG_4,inattention_LAG_4,discriminability_error_LAG_4,difficulty_error_LAG_4,guessing_error_LAG_4,inattention_error_LAG_4,auc_roc_LAG_4,optimal_threshold_LAG_4,tpr_LAG_4,tnr_LAG_4,student_mean_accuracy_LAG_4,discriminability_LAG_5,difficulty_LAG_5,guessing_LAG_5,inattention_LAG_5,discriminability_error_LAG_5,difficulty_error_LAG_5,guessing_error_LAG_5,inattention_error_LAG_5,auc_roc_LAG_5,optimal_threshold_LAG_5,tpr_LAG_5,tnr_LAG_5,student_mean_accuracy_LAG_5,discriminability_LAG_6,difficulty_LAG_6,guessing_LAG_6,inattention_LAG_6,discriminability_error_LAG_6,difficulty_error_LAG_6,guessing_error_LAG_6,inattention_error_LAG_6,auc_roc_LAG_6,optimal_threshold_LAG_6,tpr_LAG_6,tnr_LAG_6,student_mean_accuracy_LAG_6,discriminability_LAG_7,difficulty_LAG_7,guessing_LAG_7,inattention_LAG_7,discriminability_error_LAG_7,difficulty_error_LAG_7,guessing_error_LAG_7,inattention_error_LAG_7,auc_roc_LAG_7,optimal_threshold_LAG_7,tpr_LAG_7,tnr_LAG_7,student_mean_accuracy_LAG_7,discriminability_LAG_8,difficulty_LAG_8,guessing_LAG_8,inattention_LAG_8,discriminability_error_LAG_8,difficulty_error_LAG_8,guessing_error_LAG_8,inattention_error_LAG_8,auc_roc_LAG_8,optimal_threshold_LAG_8,tpr_LAG_8,tnr_LAG_8,student_mean_accuracy_LAG_8,discriminability_LAG_9,difficulty_LAG_9,guessing_LAG_9,inattention_LAG_9,discriminability_error_LAG_9,difficulty_error_LAG_9,guessing_error_LAG_9,inattention_error_LAG_9,auc_roc_LAG_9,optimal_threshold_LAG_9,tpr_LAG_9,tnr_LAG_9,student_mean_accuracy_LAG_9,discriminability_LAG_10,difficulty_LAG_10,guessing_LAG_10,inattention_LAG_10,discriminability_error_LAG_10,difficulty_error_LAG_10,guessing_error_LAG_10,inattention_error_LAG_10,auc_roc_LAG_10,optimal_threshold_LAG_10,tpr_LAG_10,tnr_LAG_10,student_mean_accuracy_LAG_10,LOG_sample_size,LOG_sample_size_LAG_1,LOG_sample_size_LAG_2,LOG_sample_size_LAG_3,LOG_sample_size_LAG_4,LOG_sample_size_LAG_5,LOG_sample_size_LAG_6,LOG_sample_

In [22]:
test_input = {
  "skillId": "afa3bfaa479de311b77c005056801da1",
  "questionId": "ff9277e80b8b",
  "eventTime": "2024-11-22 16:20:01.715000+00:00",
  "questionIdsHistory": [
    "ff9277e80b8b",
    "7fa56d98a226",
    "e233aaa276c5"
  ],
  "correctnessHistory": [
    100,
    100,
    100
  ],
  "durationSecondsHistory": [
    6.0,
    5.0,
    4.0
  ],
  "eventTimesHistory": [
    "2024-11-22T16:20:01.000Z",
    "2024-11-22T16:20:01.000Z",
    "2024-11-22T16:20:01.000Z"
  ]
}

#check output of single iference test

if debugInputs:
    results, processed_inputs = run_inference(test_input)
else:
    results = run_inference(test_input)


results

,item_prediction,item_prediction_error,item_prediction_confidence,skill_prediction,skill_prediction_error,skill_prediction_confidence
0,0.969364,0.054324,99.999194,0.979191,0.037919,99.999194


In [23]:
if debugInputs:
    display(processed_inputs)

,CORRECTNESS_LAG_1,CORRECTNESS_LAG_2,CORRECTNESS_LAG_3,CORRECTNESS_LAG_4,CORRECTNESS_LAG_5,CORRECTNESS_LAG_6,CORRECTNESS_LAG_7,CORRECTNESS_LAG_8,CORRECTNESS_LAG_9,CORRECTNESS_LAG_10,DURATIONSECONDS_LAG_1,DURATIONSECONDS_LAG_2,DURATIONSECONDS_LAG_3,DURATIONSECONDS_LAG_4,DURATIONSECONDS_LAG_5,DURATIONSECONDS_LAG_6,DURATIONSECONDS_LAG_7,DURATIONSECONDS_LAG_8,DURATIONSECONDS_LAG_9,DURATIONSECONDS_LAG_10,OCCURREDAT_DIFF_1,OCCURREDAT_DIFF_2,OCCURREDAT_DIFF_3,OCCURREDAT_DIFF_4,OCCURREDAT_DIFF_5,OCCURREDAT_DIFF_6,OCCURREDAT_DIFF_7,OCCURREDAT_DIFF_8,OCCURREDAT_DIFF_9,OCCURREDAT_DIFF_10,LOG_OCCURREDAT_DIFF_1,LOG_OCCURREDAT_DIFF_2,LOG_OCCURREDAT_DIFF_3,LOG_OCCURREDAT_DIFF_4,LOG_OCCURREDAT_DIFF_5,LOG_OCCURREDAT_DIFF_6,LOG_OCCURREDAT_DIFF_7,LOG_OCCURREDAT_DIFF_8,LOG_OCCURREDAT_DIFF_9,LOG_OCCURREDAT_DIFF_10,LOG_DURATIONSECONDS_LAG_1,LOG_DURATIONSECONDS_LAG_2,LOG_DURATIONSECONDS_LAG_3,LOG_DURATIONSECONDS_LAG_4,LOG_DURATIONSECONDS_LAG_5,LOG_DURATIONSECONDS_LAG_6,LOG_DURATIONSECONDS_LAG_7,LOG_DURATIONSECONDS_LAG_8,LOG_DURATIONSECONDS_LAG_9,LOG_DURATIONSECONDS_LAG_10,discriminability,difficulty,guessing,inattention,discriminability_error,difficulty_error,guessing_error,inattention_error,auc_roc,optimal_threshold,tpr,tnr,skill_optimal_threshold,student_mean_accuracy,discriminability_LAG_1,difficulty_LAG_1,guessing_LAG_1,inattention_LAG_1,discriminability_error_LAG_1,difficulty_error_LAG_1,guessing_error_LAG_1,inattention_error_LAG_1,auc_roc_LAG_1,optimal_threshold_LAG_1,tpr_LAG_1,tnr_LAG_1,student_mean_accuracy_LAG_1,discriminability_LAG_2,difficulty_LAG_2,guessing_LAG_2,inattention_LAG_2,discriminability_error_LAG_2,difficulty_error_LAG_2,guessing_error_LAG_2,inattention_error_LAG_2,auc_roc_LAG_2,optimal_threshold_LAG_2,tpr_LAG_2,tnr_LAG_2,student_mean_accuracy_LAG_2,discriminability_LAG_3,difficulty_LAG_3,guessing_LAG_3,inattention_LAG_3,discriminability_error_LAG_3,difficulty_error_LAG_3,guessing_error_LAG_3,inattention_error_LAG_3,auc_roc_LAG_3,optimal_threshold_LAG_3,tpr_LAG_3,tnr_LAG_3,student_mean_accuracy_LAG_3,discriminability_LAG_4,difficulty_LAG_4,guessing_LAG_4,inattention_LAG_4,discriminability_error_LAG_4,difficulty_error_LAG_4,guessing_error_LAG_4,inattention_error_LAG_4,auc_roc_LAG_4,optimal_threshold_LAG_4,tpr_LAG_4,tnr_LAG_4,student_mean_accuracy_LAG_4,discriminability_LAG_5,difficulty_LAG_5,guessing_LAG_5,inattention_LAG_5,discriminability_error_LAG_5,difficulty_error_LAG_5,guessing_error_LAG_5,inattention_error_LAG_5,auc_roc_LAG_5,optimal_threshold_LAG_5,tpr_LAG_5,tnr_LAG_5,student_mean_accuracy_LAG_5,discriminability_LAG_6,difficulty_LAG_6,guessing_LAG_6,inattention_LAG_6,discriminability_error_LAG_6,difficulty_error_LAG_6,guessing_error_LAG_6,inattention_error_LAG_6,auc_roc_LAG_6,optimal_threshold_LAG_6,tpr_LAG_6,tnr_LAG_6,student_mean_accuracy_LAG_6,discriminability_LAG_7,difficulty_LAG_7,guessing_LAG_7,inattention_LAG_7,discriminability_error_LAG_7,difficulty_error_LAG_7,guessing_error_LAG_7,inattention_error_LAG_7,auc_roc_LAG_7,optimal_threshold_LAG_7,tpr_LAG_7,tnr_LAG_7,student_mean_accuracy_LAG_7,discriminability_LAG_8,difficulty_LAG_8,guessing_LAG_8,inattention_LAG_8,discriminability_error_LAG_8,difficulty_error_LAG_8,guessing_error_LAG_8,inattention_error_LAG_8,auc_roc_LAG_8,optimal_threshold_LAG_8,tpr_LAG_8,tnr_LAG_8,student_mean_accuracy_LAG_8,discriminability_LAG_9,difficulty_LAG_9,guessing_LAG_9,inattention_LAG_9,discriminability_error_LAG_9,difficulty_error_LAG_9,guessing_error_LAG_9,inattention_error_LAG_9,auc_roc_LAG_9,optimal_threshold_LAG_9,tpr_LAG_9,tnr_LAG_9,student_mean_accuracy_LAG_9,discriminability_LAG_10,difficulty_LAG_10,guessing_LAG_10,inattention_LAG_10,discriminability_error_LAG_10,difficulty_error_LAG_10,guessing_error_LAG_10,inattention_error_LAG_10,auc_roc_LAG_10,optimal_threshold_LAG_10,tpr_LAG_10,tnr_LAG_10,student_mean_accuracy_LAG_10,LOG_sample_size,LOG_sample_size_LAG_1,LOG_sample_size_LAG_2,LOG_sample_size_LAG_3,LOG_sample_size_LAG_4,LOG_sample_size_LAG_5,LOG_sample_size_LAG_6,LOG_sample_

In [24]:
log_text = ""
log_text = "{'CORRECTNESS_LAG_1': 0.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 1.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 1.0, 'CORRECTNESS_LAG_7': 1.0, 'CORRECTNESS_LAG_8': 1.0, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 20.0, 'DURATIONSECONDS_LAG_2': 10.0, 'DURATIONSECONDS_LAG_3': 10.0, 'DURATIONSECONDS_LAG_4': 7.0, 'DURATIONSECONDS_LAG_5': 15.0, 'DURATIONSECONDS_LAG_6': 16.0, 'DURATIONSECONDS_LAG_7': 12.0, 'DURATIONSECONDS_LAG_8': 28.0, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.007031944444444445, 'OCCURREDAT_DIFF_2': 0.006427222222222223, 'OCCURREDAT_DIFF_3': 0.0036888888888888887, 'OCCURREDAT_DIFF_4': 0.0033069444444444444, 'OCCURREDAT_DIFF_5': 0.0030475, 'OCCURREDAT_DIFF_6': 0.0051491666666666665, 'OCCURREDAT_DIFF_7': 0.0060308333333333325, 'OCCURREDAT_DIFF_8': 0.0039375, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -4.957292019022328, 'LOG_OCCURREDAT_DIFF_2': -5.047212836837504, 'LOG_OCCURREDAT_DIFF_3': -5.602429980395914, 'LOG_OCCURREDAT_DIFF_4': -5.7117306445053755, 'LOG_OCCURREDAT_DIFF_5': -5.793433696608847, 'LOG_OCCURREDAT_DIFF_6': -5.268920389697249, 'LOG_OCCURREDAT_DIFF_7': -5.110870079892101, 'LOG_OCCURREDAT_DIFF_8': -5.537209274830386, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 2.995732273553991, 'LOG_DURATIONSECONDS_LAG_2': 2.3025850929940455, 'LOG_DURATIONSECONDS_LAG_3': 2.3025850929940455, 'LOG_DURATIONSECONDS_LAG_4': 1.9459101490553132, 'LOG_DURATIONSECONDS_LAG_5': 2.70805020110221, 'LOG_DURATIONSECONDS_LAG_6': 2.772588722239781, 'LOG_DURATIONSECONDS_LAG_7': 2.4849066497880004, 'LOG_DURATIONSECONDS_LAG_8': 3.332204510175204, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 99.99999999823008, 'difficulty': -0.0530587846425355, 'guessing': 3.730905617089709e-12, 'inattention': 0.999999999999906, 'discriminability_error': 4.325387690389334, 'difficulty_error': 0.0009463099413177, 'guessing_error': 0.0344904443080715, 'inattention_error': 0.0022651768021817, 'auc_roc': 0.965998645156742, 'optimal_threshold': 0.9445335525266142, 'tpr': 0.9134034459610652, 'tnr': 0.8972602739726028, 'skill_optimal_threshold': 0.8110435877206065, 'student_mean_accuracy': 0.9683640303358612, 'discriminability_LAG_1': 35.980351346211386, 'difficulty_LAG_1': -0.0537010088659765, 'guessing_LAG_1': 0.0031287821729272, 'inattention_LAG_1': 0.9889591898367812, 'discriminability_error_LAG_1': 2.7447413184588187, 'difficulty_error_LAG_1': 0.0052373921980532, 'guessing_error_LAG_1': 0.0851599724004755, 'inattention_error_LAG_1': 0.0069439363413818, 'auc_roc_LAG_1': 0.8067953884315229, 'optimal_threshold_LAG_1': 0.9435624099878296, 'tpr_LAG_1': 0.4726618705035971, 'tnr_LAG_1': 0.9896907216494846, 'student_mean_accuracy_LAG_1': 0.8775252525252525, 'discriminability_LAG_2': 67.57665477667021, 'difficulty_LAG_2': -0.0424570818931537, 'guessing_LAG_2': 0.0733753853647509, 'inattention_LAG_2': 0.9999999999999986, 'discriminability_error_LAG_2': 3.949315833791944, 'difficulty_error_LAG_2': 0.0017796091460269, 'guessing_error_LAG_2': 0.0416986757411968, 'inattention_error_LAG_2': 0.0049766374461101, 'auc_roc_LAG_2': 0.9042310165706196, 'optimal_threshold_LAG_2': 0.9309694345129784, 'tpr_LAG_2': 0.7338007736943907, 'tnr_LAG_2': 0.9149484536082474, 'student_mean_accuracy_LAG_2': 0.9142351900972592, 'discriminability_LAG_3': 99.99999999881672, 'difficulty_LAG_3': -0.0484685887269165, 'guessing_LAG_3': 0.0475193105644552, 'inattention_LAG_3': 0.9995777590544302, 'discriminability_error_LAG_3': 4.8767146571109405, 'difficulty_error_LAG_3': 0.0009356769500218, 'guessing_error_LAG_3': 0.0282250476722072, 'inattention_error_LAG_3': 0.0030912177375323, 'auc_roc_LAG_3': 0.9490072264518536, 'optimal_threshold_LAG_3': 0.951224170552168, 'tpr_LAG_3': 0.858357628765792, 'tnr_LAG_3': 0.8767772511848342, 'student_mean_accuracy_LAG_3': 0.9512364224636006, 'discriminability_LAG_4': 93.91073022177648, 'difficulty_LAG_4': -0.0474939572282289, 'guessing_LAG_4': 0.0230484795157966, 'inattention_LAG_4': 0.999999999999998, 'discriminability_error_LAG_4': 4.561703356137178, 'difficulty_error_LAG_4': 0.000959170463215, 'guessing_error_LAG_4': 0.0288091917186238, 'inattention_error_LAG_4': 0.0035110919156675, 'auc_roc_LAG_4': 0.9553907537514096, 'optimal_threshold_LAG_4': 0.9449233810533944, 'tpr_LAG_4': 0.8495966692688004, 'tnr_LAG_4': 0.9458333333333332, 'student_mean_accuracy_LAG_4': 0.94121969140338, 'discriminability_LAG_5': 61.24621112425612, 'difficulty_LAG_5': -0.0502108587597412, 'guessing_LAG_5': 7.996020747556048e-17, 'inattention_LAG_5': 0.996444309599566, 'discriminability_error_LAG_5': 3.96489907699478, 'difficulty_error_LAG_5': 0.0023327875708825, 'guessing_error_LAG_5': 0.0540346302092938, 'inattention_error_LAG_5': 0.0056488254936373, 'auc_roc_LAG_5': 0.8908214927323207, 'optimal_threshold_LAG_5': 0.93172536034524, 'tpr_LAG_5': 0.716105550500455, 'tnr_LAG_5': 0.891025641025641, 'student_mean_accuracy_LAG_5': 0.913549459684123, 'discriminability_LAG_6': 99.99999995962833, 'difficulty_LAG_6': -0.0463147037643052, 'guessing_LAG_6': 0.1051746476125233, 'inattention_LAG_6': 0.9999999999998468, 'discriminability_error_LAG_6': 5.74547821027573, 'difficulty_error_LAG_6': 0.0009800792024589, 'guessing_error_LAG_6': 0.0274705834541801, 'inattention_error_LAG_6': 0.0037923892301089, 'auc_roc_LAG_6': 0.9660262179453244, 'optimal_threshold_LAG_6': 0.9313171319928, 'tpr_LAG_6': 0.883344803854095, 'tnr_LAG_6': 0.9467455621301776, 'student_mean_accuracy_LAG_6': 0.945040650406504, 'discriminability_LAG_7': 20.47167528144286, 'difficulty_LAG_7': -0.0317006143011296, 'guessing_LAG_7': 8.462389127616201e-16, 'inattention_LAG_7': 0.9999999987317566, 'discriminability_error_LAG_7': 3.483226213007768, 'difficulty_error_LAG_7': 0.0152065628549162, 'guessing_error_LAG_7': 0.1526383975011337, 'inattention_error_LAG_7': 0.0171377966783443, 'auc_roc_LAG_7': 0.7937800462935486, 'optimal_threshold_LAG_7': 0.8611196667005327, 'tpr_LAG_7': 0.4596059113300492, 'tnr_LAG_7': 1.0, 'student_mean_accuracy_LAG_7': 0.731004681310767, 'discriminability_LAG_8': 99.99999977003948, 'difficulty_LAG_8': -0.0225685885765456, 'guessing_LAG_8': 0.1435735856010376, 'inattention_LAG_8': 0.999999999999096, 'discriminability_error_LAG_8': 9.52513695427783, 'difficulty_error_LAG_8': 0.0013069323582309, 'guessing_error_LAG_8': 0.0350851797279684, 'inattention_error_LAG_8': 0.0099471496213624, 'auc_roc_LAG_8': 0.9310168785757882, 'optimal_threshold_LAG_8': 0.8119286514631636, 'tpr_LAG_8': 0.8219278881530537, 'tnr_LAG_8': 0.8992248062015504, 'student_mean_accuracy_LAG_8': 0.8404452690166976, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 8.437067146936952, 'LOG_sample_size_LAG_1': 8.466320861042481, 'LOG_sample_size_LAG_2': 8.41715183723601, 'LOG_sample_size_LAG_3': 8.372629740224884, 'LOG_sample_size_LAG_4': 8.314587291319576, 'LOG_sample_size_LAG_5': 8.191186004642788, 'LOG_sample_size_LAG_6': 8.031060180240617, 'LOG_sample_size_LAG_7': 7.9291264873067995, 'LOG_sample_size_LAG_8': 7.388327859577107, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': 0.0311324202894309, 'optimal_threshold_spread': 0.1392955190890044, 'student_mean_accuracy_spread': 0.22023174115283362, 'tnr_spread': 0.12322274881516582, 'tpr_spread': 0.4237388925240458, 'mean_auc_roc': 0.8996336275940484, 'mean_discriminability': 72.3982028098552, 'mean_discriminability_error': 4.856401952506873, 'mean_difficulty_error': 0.003592276342975675, 'mean_guessing_error': 0.05664020980313492, 'mean_inattention_error': 0.006881130558018075, 'num_responses': 8.0}"
log_text = "{'CORRECTNESS_LAG_1': nan, 'CORRECTNESS_LAG_2': nan, 'CORRECTNESS_LAG_3': nan, 'CORRECTNESS_LAG_4': nan, 'CORRECTNESS_LAG_5': nan, 'CORRECTNESS_LAG_6': nan, 'CORRECTNESS_LAG_7': nan, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': nan, 'DURATIONSECONDS_LAG_2': nan, 'DURATIONSECONDS_LAG_3': nan, 'DURATIONSECONDS_LAG_4': nan, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': nan, 'OCCURREDAT_DIFF_2': nan, 'OCCURREDAT_DIFF_3': nan, 'OCCURREDAT_DIFF_4': nan, 'OCCURREDAT_DIFF_5': nan, 'OCCURREDAT_DIFF_6': nan, 'OCCURREDAT_DIFF_7': nan, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': nan, 'LOG_OCCURREDAT_DIFF_2': nan, 'LOG_OCCURREDAT_DIFF_3': nan, 'LOG_OCCURREDAT_DIFF_4': nan, 'LOG_OCCURREDAT_DIFF_5': nan, 'LOG_OCCURREDAT_DIFF_6': nan, 'LOG_OCCURREDAT_DIFF_7': nan, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': nan, 'LOG_DURATIONSECONDS_LAG_2': nan, 'LOG_DURATIONSECONDS_LAG_3': nan, 'LOG_DURATIONSECONDS_LAG_4': nan, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 99.99999999995248, 'difficulty': 0.0331485069212573, 'guessing': 4.319358647443306e-19, 'inattention': 0.9999999999664682, 'discriminability_error': 18.60556905222149, 'difficulty_error': 0.0022020586623096, 'guessing_error': 0.0575736119511004, 'inattention_error': 0.0349087059685982, 'auc_roc': 0.9365463467096862, 'optimal_threshold': 0.6178543516176616, 'tpr': 0.8830409356725146, 'tnr': 0.8620689655172413, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.6627906976744186, 'discriminability_LAG_1': nan, 'difficulty_LAG_1': nan, 'guessing_LAG_1': nan, 'inattention_LAG_1': nan, 'discriminability_error_LAG_1': nan, 'difficulty_error_LAG_1': nan, 'guessing_error_LAG_1': nan, 'inattention_error_LAG_1': nan, 'auc_roc_LAG_1': nan, 'optimal_threshold_LAG_1': nan, 'tpr_LAG_1': nan, 'tnr_LAG_1': nan, 'student_mean_accuracy_LAG_1': nan, 'discriminability_LAG_2': nan, 'difficulty_LAG_2': nan, 'guessing_LAG_2': nan, 'inattention_LAG_2': nan, 'discriminability_error_LAG_2': nan, 'difficulty_error_LAG_2': nan, 'guessing_error_LAG_2': nan, 'inattention_error_LAG_2': nan, 'auc_roc_LAG_2': nan, 'optimal_threshold_LAG_2': nan, 'tpr_LAG_2': nan, 'tnr_LAG_2': nan, 'student_mean_accuracy_LAG_2': nan, 'discriminability_LAG_3': nan, 'difficulty_LAG_3': nan, 'guessing_LAG_3': nan, 'inattention_LAG_3': nan, 'discriminability_error_LAG_3': nan, 'difficulty_error_LAG_3': nan, 'guessing_error_LAG_3': nan, 'inattention_error_LAG_3': nan, 'auc_roc_LAG_3': nan, 'optimal_threshold_LAG_3': nan, 'tpr_LAG_3': nan, 'tnr_LAG_3': nan, 'student_mean_accuracy_LAG_3': nan, 'discriminability_LAG_4': nan, 'difficulty_LAG_4': nan, 'guessing_LAG_4': nan, 'inattention_LAG_4': nan, 'discriminability_error_LAG_4': nan, 'difficulty_error_LAG_4': nan, 'guessing_error_LAG_4': nan, 'inattention_error_LAG_4': nan, 'auc_roc_LAG_4': nan, 'optimal_threshold_LAG_4': nan, 'tpr_LAG_4': nan, 'tnr_LAG_4': nan, 'student_mean_accuracy_LAG_4': nan, 'discriminability_LAG_5': nan, 'difficulty_LAG_5': nan, 'guessing_LAG_5': nan, 'inattention_LAG_5': nan, 'discriminability_error_LAG_5': nan, 'difficulty_error_LAG_5': nan, 'guessing_error_LAG_5': nan, 'inattention_error_LAG_5': nan, 'auc_roc_LAG_5': nan, 'optimal_threshold_LAG_5': nan, 'tpr_LAG_5': nan, 'tnr_LAG_5': nan, 'student_mean_accuracy_LAG_5': nan, 'discriminability_LAG_6': nan, 'difficulty_LAG_6': nan, 'guessing_LAG_6': nan, 'inattention_LAG_6': nan, 'discriminability_error_LAG_6': nan, 'difficulty_error_LAG_6': nan, 'guessing_error_LAG_6': nan, 'inattention_error_LAG_6': nan, 'auc_roc_LAG_6': nan, 'optimal_threshold_LAG_6': nan, 'tpr_LAG_6': nan, 'tnr_LAG_6': nan, 'student_mean_accuracy_LAG_6': nan, 'discriminability_LAG_7': nan, 'difficulty_LAG_7': nan, 'guessing_LAG_7': nan, 'inattention_LAG_7': nan, 'discriminability_error_LAG_7': nan, 'difficulty_error_LAG_7': nan, 'guessing_error_LAG_7': nan, 'inattention_error_LAG_7': nan, 'auc_roc_LAG_7': nan, 'optimal_threshold_LAG_7': nan, 'tpr_LAG_7': nan, 'tnr_LAG_7': nan, 'student_mean_accuracy_LAG_7': nan, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 5.552959584921617, 'LOG_sample_size_LAG_1': nan, 'LOG_sample_size_LAG_2': nan, 'LOG_sample_size_LAG_3': nan, 'LOG_sample_size_LAG_4': nan, 'LOG_sample_size_LAG_5': nan, 'LOG_sample_size_LAG_6': nan, 'LOG_sample_size_LAG_7': nan, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': nan, 'optimal_threshold_spread': nan, 'student_mean_accuracy_spread': nan, 'tnr_spread': nan, 'tpr_spread': nan, 'mean_auc_roc': nan, 'mean_discriminability': nan, 'mean_discriminability_error': nan, 'mean_difficulty_error': nan, 'mean_guessing_error': nan, 'mean_inattention_error': nan, 'num_responses': 0.0}"
log_text = "{'CORRECTNESS_LAG_1': 0.0, 'CORRECTNESS_LAG_2': 0.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 1.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 1.0, 'CORRECTNESS_LAG_7': 0.0, 'CORRECTNESS_LAG_8': 0.0, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 82.0, 'DURATIONSECONDS_LAG_2': 184.0, 'DURATIONSECONDS_LAG_3': 234.0, 'DURATIONSECONDS_LAG_4': 163.0, 'DURATIONSECONDS_LAG_5': 130.0, 'DURATIONSECONDS_LAG_6': 76.0, 'DURATIONSECONDS_LAG_7': 43.0, 'DURATIONSECONDS_LAG_8': 77.0, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.026307, 'OCCURREDAT_DIFF_2': 0.024746, 'OCCURREDAT_DIFF_3': 0.051342, 'OCCURREDAT_DIFF_4': 216.036479, 'OCCURREDAT_DIFF_5': 0.047071, 'OCCURREDAT_DIFF_6': 0.036434, 'OCCURREDAT_DIFF_7': 0.021784, 'OCCURREDAT_DIFF_8': 0.012984, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -3.6379202155168087, 'LOG_OCCURREDAT_DIFF_2': -3.699091419190353, 'LOG_OCCURREDAT_DIFF_3': -2.969246148318164, 'LOG_OCCURREDAT_DIFF_4': 5.375447277684083, 'LOG_OCCURREDAT_DIFF_5': -3.0560981788404336, 'LOG_OCCURREDAT_DIFF_6': -3.3122528743735, 'LOG_OCCURREDAT_DIFF_7': -3.8265795236106785, 'LOG_OCCURREDAT_DIFF_8': -4.344037448769846, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 4.406719247264253, 'LOG_DURATIONSECONDS_LAG_2': 5.214935757608986, 'LOG_DURATIONSECONDS_LAG_3': 5.455321115357701, 'LOG_DURATIONSECONDS_LAG_4': 5.093750200806762, 'LOG_DURATIONSECONDS_LAG_5': 4.867534450455582, 'LOG_DURATIONSECONDS_LAG_6': 4.330733340286331, 'LOG_DURATIONSECONDS_LAG_7': 3.7612001156935624, 'LOG_DURATIONSECONDS_LAG_8': 4.343805421853684, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 7.2815231885235105, 'difficulty': nan, 'guessing': 6.047929286861074e-15, 'inattention': 0.9999999999786012, 'discriminability_error': 2.537920314021872, 'difficulty_error': 0.0469598514285472, 'guessing_error': 0.119021470725214, 'inattention_error': 0.087897415874614, 'auc_roc': 0.8159259259259259, 'optimal_threshold': 0.6613106163446428, 'tpr': 0.7847222222222222, 'tnr': 0.7466666666666666, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.6575342465753424, 'discriminability_LAG_1': 23.52989611194068, 'difficulty_LAG_1': nan, 'guessing_LAG_1': 0.0010096569682835, 'inattention_LAG_1': 0.9879832440712718, 'discriminability_error_LAG_1': 6.388888733430298, 'difficulty_error_LAG_1': 0.0140142248198231, 'guessing_error_LAG_1': 0.094206255832491, 'inattention_error_LAG_1': 0.0522917713676038, 'auc_roc_LAG_1': 0.8949218750000001, 'optimal_threshold_LAG_1': 0.5901403373071329, 'tpr_LAG_1': 0.8916666666666667, 'tnr_LAG_1': 0.75, 'student_mean_accuracy_LAG_1': 0.6521739130434783, 'discriminability_LAG_2': 11.297346448027508, 'difficulty_LAG_2': nan, 'guessing_LAG_2': 5.949014493459207e-20, 'inattention_LAG_2': 0.9999999999999996, 'discriminability_error_LAG_2': 6.905612379827064, 'difficulty_error_LAG_2': 0.0762354887823375, 'guessing_error_LAG_2': 0.3722741268929199, 'inattention_error_LAG_2': 0.143245996050876, 'auc_roc_LAG_2': 0.7580280416101312, 'optimal_threshold_LAG_2': 0.5792058422516965, 'tpr_LAG_2': 0.8507462686567164, 'tnr_LAG_2': 0.5454545454545454, 'student_mean_accuracy_LAG_2': 0.67, 'discriminability_LAG_3': 10.389481329612922, 'difficulty_LAG_3': nan, 'guessing_LAG_3': 4.133369078308072e-21, 'inattention_LAG_3': 0.9931753464004264, 'discriminability_error_LAG_3': 5.279353389551476, 'difficulty_error_LAG_3': 0.0664911133833362, 'guessing_error_LAG_3': 0.2193600923978763, 'inattention_error_LAG_3': 0.1080802923343528, 'auc_roc_LAG_3': 0.8137755102040816, 'optimal_threshold_LAG_3': 0.6561116563888924, 'tpr_LAG_3': 0.9142857142857144, 'tnr_LAG_3': 0.6071428571428572, 'student_mean_accuracy_LAG_3': 0.7142857142857143, 'discriminability_LAG_4': 5.801981526569862, 'difficulty_LAG_4': nan, 'guessing_LAG_4': 0.1967457663531708, 'inattention_LAG_4': 0.9999999999929234, 'discriminability_error_LAG_4': 3.641241809602025, 'difficulty_error_LAG_4': 0.0939929566923505, 'guessing_error_LAG_4': 0.1956943231292651, 'inattention_error_LAG_4': 0.1250333983399777, 'auc_roc_LAG_4': 0.745536903810285, 'optimal_threshold_LAG_4': 0.8225902096110265, 'tpr_LAG_4': 0.381294964028777, 'tnr_LAG_4': 1.0, 'student_mean_accuracy_LAG_4': 0.7202072538860104, 'discriminability_LAG_5': 15.11037049017176, 'difficulty_LAG_5': nan, 'guessing_LAG_5': 1.5873110080669933e-21, 'inattention_LAG_5': 1.0, 'discriminability_error_LAG_5': 4.524469801055274, 'difficulty_error_LAG_5': 0.0223965291464581, 'guessing_error_LAG_5': 0.11622365484548, 'inattention_error_LAG_5': 0.0538796352477706, 'auc_roc_LAG_5': 0.8776836158192091, 'optimal_threshold_LAG_5': 0.639750014806599, 'tpr_LAG_5': 0.7666666666666667, 'tnr_LAG_5': 0.8135593220338984, 'student_mean_accuracy_LAG_5': 0.6703910614525139, 'discriminability_LAG_6': 10.220339179544194, 'difficulty_LAG_6': nan, 'guessing_LAG_6': 8.223052679749138e-14, 'inattention_LAG_6': 0.9999999999999964, 'discriminability_error_LAG_6': 3.591631357947908, 'difficulty_error_LAG_6': 0.0548880836198887, 'guessing_error_LAG_6': 0.2049464025978156, 'inattention_error_LAG_6': 0.0524719085933737, 'auc_roc_LAG_6': 0.8250474383301708, 'optimal_threshold_LAG_6': 0.7968058350135704, 'tpr_LAG_6': 0.7612903225806451, 'tnr_LAG_6': 0.7647058823529411, 'student_mean_accuracy_LAG_6': 0.8201058201058201, 'discriminability_LAG_7': 3.901865602329048, 'difficulty_LAG_7': nan, 'guessing_LAG_7': 8.510172068641542e-12, 'inattention_LAG_7': 0.9999999999997824, 'discriminability_error_LAG_7': 2.414701833094194, 'difficulty_error_LAG_7': 0.2124286461198327, 'guessing_error_LAG_7': 0.3242676803889134, 'inattention_error_LAG_7': 0.1286217293337347, 'auc_roc_LAG_7': 0.7996941896024465, 'optimal_threshold_LAG_7': 0.8280211913445781, 'tpr_LAG_7': 0.5688073394495413, 'tnr_LAG_7': 0.9166666666666666, 'student_mean_accuracy_LAG_7': 0.8195488721804511, 'discriminability_LAG_8': 99.99999996013511, 'difficulty_LAG_8': nan, 'guessing_LAG_8': 0.1482827461027896, 'inattention_LAG_8': 0.9999999414528732, 'discriminability_error_LAG_8': 38.488366351154106, 'difficulty_error_LAG_8': 0.0047140610562256, 'guessing_error_LAG_8': 0.0406062966861051, 'inattention_error_LAG_8': 0.0551593852946379, 'auc_roc_LAG_8': 0.9020270270270272, 'optimal_threshold_LAG_8': 0.6545208818965591, 'tpr_LAG_8': 0.78125, 'tnr_LAG_8': 1.0, 'student_mean_accuracy_LAG_8': 0.463768115942029, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 5.389071729816501, 'LOG_sample_size_LAG_1': 5.214935757608986, 'LOG_sample_size_LAG_2': 4.605170185988091, 'LOG_sample_size_LAG_3': 4.584967478670571, 'LOG_sample_size_LAG_4': 5.262690188904886, 'LOG_sample_size_LAG_5': 5.187385805840755, 'LOG_sample_size_LAG_6': 5.241747015059643, 'LOG_sample_size_LAG_7': 4.890349128221754, 'LOG_sample_size_LAG_8': 4.927253685157204, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': nan, 'optimal_threshold_spread': 0.2488153490928816, 'student_mean_accuracy_spread': 0.3563377041637911, 'tnr_spread': 0.4545454545454546, 'tpr_spread': 0.5329907502569373, 'mean_auc_roc': 0.827089325175419, 'mean_discriminability': 22.531410081041386, 'mean_discriminability_error': 8.904283206957793, 'mean_difficulty_error': 0.06814513795253155, 'mean_guessing_error': 0.1959473540963583, 'mean_inattention_error': 0.0898480145702909, 'num_responses': 8.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 0.0, 'CORRECTNESS_LAG_4': nan, 'CORRECTNESS_LAG_5': nan, 'CORRECTNESS_LAG_6': nan, 'CORRECTNESS_LAG_7': nan, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 243.0, 'DURATIONSECONDS_LAG_2': nan, 'DURATIONSECONDS_LAG_3': nan, 'DURATIONSECONDS_LAG_4': nan, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.01639, 'OCCURREDAT_DIFF_2': 287.902926, 'OCCURREDAT_DIFF_3': 0.003602, 'OCCURREDAT_DIFF_4': nan, 'OCCURREDAT_DIFF_5': nan, 'OCCURREDAT_DIFF_6': nan, 'OCCURREDAT_DIFF_7': nan, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -4.111083886226399, 'LOG_OCCURREDAT_DIFF_2': 5.662623360817613, 'LOG_OCCURREDAT_DIFF_3': -5.626266032228372, 'LOG_OCCURREDAT_DIFF_4': nan, 'LOG_OCCURREDAT_DIFF_5': nan, 'LOG_OCCURREDAT_DIFF_6': nan, 'LOG_OCCURREDAT_DIFF_7': nan, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 5.493061443340548, 'LOG_DURATIONSECONDS_LAG_2': nan, 'LOG_DURATIONSECONDS_LAG_3': nan, 'LOG_DURATIONSECONDS_LAG_4': nan, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': nan, 'difficulty': nan, 'guessing': nan, 'inattention': nan, 'discriminability_error': nan, 'difficulty_error': nan, 'guessing_error': nan, 'inattention_error': nan, 'auc_roc': nan, 'optimal_threshold': nan, 'tpr': nan, 'tnr': nan, 'skill_optimal_threshold': nan, 'student_mean_accuracy': nan, 'discriminability_LAG_1': nan, 'difficulty_LAG_1': nan, 'guessing_LAG_1': nan, 'inattention_LAG_1': nan, 'discriminability_error_LAG_1': nan, 'difficulty_error_LAG_1': nan, 'guessing_error_LAG_1': nan, 'inattention_error_LAG_1': nan, 'auc_roc_LAG_1': nan, 'optimal_threshold_LAG_1': nan, 'tpr_LAG_1': nan, 'tnr_LAG_1': nan, 'student_mean_accuracy_LAG_1': nan, 'discriminability_LAG_2': nan, 'difficulty_LAG_2': nan, 'guessing_LAG_2': nan, 'inattention_LAG_2': nan, 'discriminability_error_LAG_2': nan, 'difficulty_error_LAG_2': nan, 'guessing_error_LAG_2': nan, 'inattention_error_LAG_2': nan, 'auc_roc_LAG_2': nan, 'optimal_threshold_LAG_2': nan, 'tpr_LAG_2': nan, 'tnr_LAG_2': nan, 'student_mean_accuracy_LAG_2': nan, 'discriminability_LAG_3': nan, 'difficulty_LAG_3': nan, 'guessing_LAG_3': nan, 'inattention_LAG_3': nan, 'discriminability_error_LAG_3': nan, 'difficulty_error_LAG_3': nan, 'guessing_error_LAG_3': nan, 'inattention_error_LAG_3': nan, 'auc_roc_LAG_3': nan, 'optimal_threshold_LAG_3': nan, 'tpr_LAG_3': nan, 'tnr_LAG_3': nan, 'student_mean_accuracy_LAG_3': nan, 'discriminability_LAG_4': nan, 'difficulty_LAG_4': nan, 'guessing_LAG_4': nan, 'inattention_LAG_4': nan, 'discriminability_error_LAG_4': nan, 'difficulty_error_LAG_4': nan, 'guessing_error_LAG_4': nan, 'inattention_error_LAG_4': nan, 'auc_roc_LAG_4': nan, 'optimal_threshold_LAG_4': nan, 'tpr_LAG_4': nan, 'tnr_LAG_4': nan, 'student_mean_accuracy_LAG_4': nan, 'discriminability_LAG_5': nan, 'difficulty_LAG_5': nan, 'guessing_LAG_5': nan, 'inattention_LAG_5': nan, 'discriminability_error_LAG_5': nan, 'difficulty_error_LAG_5': nan, 'guessing_error_LAG_5': nan, 'inattention_error_LAG_5': nan, 'auc_roc_LAG_5': nan, 'optimal_threshold_LAG_5': nan, 'tpr_LAG_5': nan, 'tnr_LAG_5': nan, 'student_mean_accuracy_LAG_5': nan, 'discriminability_LAG_6': nan, 'difficulty_LAG_6': nan, 'guessing_LAG_6': nan, 'inattention_LAG_6': nan, 'discriminability_error_LAG_6': nan, 'difficulty_error_LAG_6': nan, 'guessing_error_LAG_6': nan, 'inattention_error_LAG_6': nan, 'auc_roc_LAG_6': nan, 'optimal_threshold_LAG_6': nan, 'tpr_LAG_6': nan, 'tnr_LAG_6': nan, 'student_mean_accuracy_LAG_6': nan, 'discriminability_LAG_7': nan, 'difficulty_LAG_7': nan, 'guessing_LAG_7': nan, 'inattention_LAG_7': nan, 'discriminability_error_LAG_7': nan, 'difficulty_error_LAG_7': nan, 'guessing_error_LAG_7': nan, 'inattention_error_LAG_7': nan, 'auc_roc_LAG_7': nan, 'optimal_threshold_LAG_7': nan, 'tpr_LAG_7': nan, 'tnr_LAG_7': nan, 'student_mean_accuracy_LAG_7': nan, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': nan, 'LOG_sample_size_LAG_1': nan, 'LOG_sample_size_LAG_2': nan, 'LOG_sample_size_LAG_3': nan, 'LOG_sample_size_LAG_4': nan, 'LOG_sample_size_LAG_5': nan, 'LOG_sample_size_LAG_6': nan, 'LOG_sample_size_LAG_7': nan, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': nan, 'optimal_threshold_spread': nan, 'student_mean_accuracy_spread': nan, 'tnr_spread': nan, 'tpr_spread': nan, 'mean_auc_roc': nan, 'mean_discriminability': nan, 'mean_discriminability_error': nan, 'mean_difficulty_error': nan, 'mean_guessing_error': nan, 'mean_inattention_error': nan, 'num_responses': 3.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 0.0, 'CORRECTNESS_LAG_4': 0.0, 'CORRECTNESS_LAG_5': nan, 'CORRECTNESS_LAG_6': nan, 'CORRECTNESS_LAG_7': nan, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 35.0, 'DURATIONSECONDS_LAG_2': 13.0, 'DURATIONSECONDS_LAG_3': 158.0, 'DURATIONSECONDS_LAG_4': 65.0, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.010717, 'OCCURREDAT_DIFF_2': 0.010162, 'OCCURREDAT_DIFF_3': 0.007917, 'OCCURREDAT_DIFF_4': 0.046469, 'OCCURREDAT_DIFF_5': nan, 'OCCURREDAT_DIFF_6': nan, 'OCCURREDAT_DIFF_7': nan, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -4.535924013251277, 'LOG_OCCURREDAT_DIFF_2': -4.589100005810597, 'LOG_OCCURREDAT_DIFF_3': -4.83874293279284, 'LOG_OCCURREDAT_DIFF_4': -3.06896985537653, 'LOG_OCCURREDAT_DIFF_5': nan, 'LOG_OCCURREDAT_DIFF_6': nan, 'LOG_OCCURREDAT_DIFF_7': nan, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 3.5553480614894135, 'LOG_DURATIONSECONDS_LAG_2': 2.5649493574615367, 'LOG_DURATIONSECONDS_LAG_3': 5.062595033026967, 'LOG_DURATIONSECONDS_LAG_4': 4.174387269895637, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 99.9999999999882, 'difficulty': 0.111574036922831, 'guessing': 0.0356095815985954, 'inattention': 1.0, 'discriminability_error': 8.804815089344459, 'difficulty_error': 0.0013342981130151, 'guessing_error': 0.007333579432506, 'inattention_error': 0.032079854665467, 'auc_roc': 0.8595546558704454, 'optimal_threshold': 0.1311314876107675, 'tpr': 0.6912280701754386, 'tnr': 0.8478021978021978, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.1353919239904988, 'discriminability_LAG_1': 21.54590076966744, 'difficulty_LAG_1': 0.0708673085049363, 'guessing_LAG_1': 0.0790491647720532, 'inattention_LAG_1': 0.9999999999999982, 'discriminability_error_LAG_1': 3.81047316001496, 'difficulty_error_LAG_1': 0.0087764767259943, 'guessing_error_LAG_1': 0.0667282490373249, 'inattention_error_LAG_1': 0.0598278233629829, 'auc_roc_LAG_1': 0.7178902115577024, 'optimal_threshold_LAG_1': 0.4971943113452648, 'tpr_LAG_1': 0.6455142231947484, 'tnr_LAG_1': 0.669327251995439, 'student_mean_accuracy_LAG_1': 0.5103294249022893, 'discriminability_LAG_2': 9.709090041960856, 'difficulty_LAG_2': -0.0045396909315414, 'guessing_LAG_2': 0.0140573254281031, 'inattention_LAG_2': 0.9999999999912764, 'discriminability_error_LAG_2': 3.102786632089864, 'difficulty_error_LAG_2': 0.0407784951064129, 'guessing_error_LAG_2': 0.1963438216051185, 'inattention_error_LAG_2': 0.0902286532644365, 'auc_roc_LAG_2': 0.6362440597420231, 'optimal_threshold_LAG_2': 0.682290165842647, 'tpr_LAG_2': 0.4409368635437882, 'tnr_LAG_2': 0.748015873015873, 'student_mean_accuracy_LAG_2': 0.6608344549125168, 'discriminability_LAG_3': 16.76948819317966, 'difficulty_LAG_3': 0.0847541601217825, 'guessing_LAG_3': 0.0800368351067536, 'inattention_LAG_3': 0.9999999999994726, 'discriminability_error_LAG_3': 3.8854513192114495, 'difficulty_error_LAG_3': 0.0141828890874568, 'guessing_error_LAG_3': 0.0782626394420638, 'inattention_error_LAG_3': 0.0884987412121089, 'auc_roc_LAG_3': 0.685558426643698, 'optimal_threshold_LAG_3': 0.464945969722891, 'tpr_LAG_3': 0.5896296296296296, 'tnr_LAG_3': 0.6757105943152455, 'student_mean_accuracy_LAG_3': 0.4658385093167702, 'discriminability_LAG_4': 99.99999999999996, 'difficulty_LAG_4': 0.0762465580938703, 'guessing_LAG_4': 0.0303315263112436, 'inattention_LAG_4': 1.0, 'discriminability_error_LAG_4': 13.304722466663446, 'difficulty_error_LAG_4': 0.0016137152535805, 'guessing_error_LAG_4': 0.0229655037123961, 'inattention_error_LAG_4': 0.0415393382421726, 'auc_roc_LAG_4': 0.9052538485461048, 'optimal_threshold_LAG_4': 0.3437484681809679, 'tpr_LAG_4': 0.7752293577981652, 'tnr_LAG_4': 0.8940677966101696, 'student_mean_accuracy_LAG_4': 0.3159420289855073, 'discriminability_LAG_5': nan, 'difficulty_LAG_5': nan, 'guessing_LAG_5': nan, 'inattention_LAG_5': nan, 'discriminability_error_LAG_5': nan, 'difficulty_error_LAG_5': nan, 'guessing_error_LAG_5': nan, 'inattention_error_LAG_5': nan, 'auc_roc_LAG_5': nan, 'optimal_threshold_LAG_5': nan, 'tpr_LAG_5': nan, 'tnr_LAG_5': nan, 'student_mean_accuracy_LAG_5': nan, 'discriminability_LAG_6': nan, 'difficulty_LAG_6': nan, 'guessing_LAG_6': nan, 'inattention_LAG_6': nan, 'discriminability_error_LAG_6': nan, 'difficulty_error_LAG_6': nan, 'guessing_error_LAG_6': nan, 'inattention_error_LAG_6': nan, 'auc_roc_LAG_6': nan, 'optimal_threshold_LAG_6': nan, 'tpr_LAG_6': nan, 'tnr_LAG_6': nan, 'student_mean_accuracy_LAG_6': nan, 'discriminability_LAG_7': nan, 'difficulty_LAG_7': nan, 'guessing_LAG_7': nan, 'inattention_LAG_7': nan, 'discriminability_error_LAG_7': nan, 'difficulty_error_LAG_7': nan, 'guessing_error_LAG_7': nan, 'inattention_error_LAG_7': nan, 'auc_roc_LAG_7': nan, 'optimal_threshold_LAG_7': nan, 'tpr_LAG_7': nan, 'tnr_LAG_7': nan, 'student_mean_accuracy_LAG_7': nan, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 7.652070746116482, 'LOG_sample_size_LAG_1': 7.490529402060711, 'LOG_sample_size_LAG_2': 7.303843225277705, 'LOG_sample_size_LAG_3': 7.278628942320682, 'LOG_sample_size_LAG_4': 6.536691597591305, 'LOG_sample_size_LAG_5': nan, 'LOG_sample_size_LAG_6': nan, 'LOG_sample_size_LAG_7': nan, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': 0.0892938510533239, 'optimal_threshold_spread': 0.33854169766167913, 'student_mean_accuracy_spread': 0.3448924259270095, 'tnr_spread': 0.2247405446147306, 'tpr_spread': 0.33429249425437696, 'mean_auc_roc': 0.736236636622382, 'mean_discriminability': 37.00611975120198, 'mean_discriminability_error': 6.0258583944949295, 'mean_difficulty_error': 0.016337894043361122, 'mean_guessing_error': 0.09107505344922583, 'mean_inattention_error': 0.07002363902042523, 'num_responses': 4.0}"
log_text = "{'CORRECTNESS_LAG_1': 0.0, 'CORRECTNESS_LAG_2': 0.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 0.0, 'CORRECTNESS_LAG_5': 0.0, 'CORRECTNESS_LAG_6': 1.0, 'CORRECTNESS_LAG_7': 0.0, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 1.0, 'DURATIONSECONDS_LAG_2': 5.0, 'DURATIONSECONDS_LAG_3': 46.0, 'DURATIONSECONDS_LAG_4': 1.0, 'DURATIONSECONDS_LAG_5': 4.0, 'DURATIONSECONDS_LAG_6': 3.0, 'DURATIONSECONDS_LAG_7': 1.0, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.007209, 'OCCURREDAT_DIFF_2': 0.0002777777777777778, 'OCCURREDAT_DIFF_3': 0.0002777777777777778, 'OCCURREDAT_DIFF_4': 0.023177, 'OCCURREDAT_DIFF_5': 0.002166, 'OCCURREDAT_DIFF_6': 0.00264, 'OCCURREDAT_DIFF_7': 0.001537, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -4.932425033559696, 'LOG_OCCURREDAT_DIFF_2': -8.1886891244442, 'LOG_OCCURREDAT_DIFF_3': -8.1886891244442, 'LOG_OCCURREDAT_DIFF_4': -3.7645948713617146, 'LOG_OCCURREDAT_DIFF_5': -6.134873130403338, 'LOG_OCCURREDAT_DIFF_6': -5.936976361823912, 'LOG_OCCURREDAT_DIFF_7': -6.477922814425678, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 0.0, 'LOG_DURATIONSECONDS_LAG_2': 1.6094379124341003, 'LOG_DURATIONSECONDS_LAG_3': 3.828641396489095, 'LOG_DURATIONSECONDS_LAG_4': 0.0, 'LOG_DURATIONSECONDS_LAG_5': 1.3862943611198906, 'LOG_DURATIONSECONDS_LAG_6': 1.0986122886681096, 'LOG_DURATIONSECONDS_LAG_7': 0.0, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 17.160758922609357, 'difficulty': -0.0167485992798112, 'guessing': 0.1076539692632642, 'inattention': 1.0, 'discriminability_error': 22.39848938967146, 'difficulty_error': 0.1016716763165803, 'guessing_error': 0.1986492739637225, 'inattention_error': 0.7455229570450075, 'auc_roc': 0.7090347923681257, 'optimal_threshold': 0.6943856021030133, 'tpr': 0.6172839506172839, 'tnr': 0.75, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.648, 'discriminability_LAG_1': 86.92839027179889, 'difficulty_LAG_1': 0.0124963351874705, 'guessing_LAG_1': 7.70420523183464e-16, 'inattention_LAG_1': 0.9999999999999982, 'discriminability_error_LAG_1': 36.053922372192126, 'difficulty_error_LAG_1': 0.0050632169183278, 'guessing_error_LAG_1': 0.1205144420113964, 'inattention_error_LAG_1': 0.1289255757417061, 'auc_roc_LAG_1': 0.866304347826087, 'optimal_threshold_LAG_1': 0.6562824544639996, 'tpr_LAG_1': 0.7333333333333333, 'tnr_LAG_1': 0.8478260869565217, 'student_mean_accuracy_LAG_1': 0.5660377358490566, 'discriminability_LAG_2': 99.99999999061068, 'difficulty_LAG_2': 0.0168779761935367, 'guessing_LAG_2': 8.28081985690192e-13, 'inattention_LAG_2': 0.9999999999825512, 'discriminability_error_LAG_2': 36.8099369786276, 'difficulty_error_LAG_2': 0.0039896529350986, 'guessing_error_LAG_2': 0.0989542528242677, 'inattention_error_LAG_2': 0.1187800712521197, 'auc_roc_LAG_2': 0.8909774436090225, 'optimal_threshold_LAG_2': 0.5139058738912498, 'tpr_LAG_2': 0.8771929824561403, 'tnr_LAG_2': 0.8214285714285714, 'student_mean_accuracy_LAG_2': 0.504424778761062, 'discriminability_LAG_3': 84.2146907660888, 'difficulty_LAG_3': 0.0024429774498881, 'guessing_LAG_3': 6.100964780328714e-31, 'inattention_LAG_3': 0.93873963109051, 'discriminability_error_LAG_3': 33.65934850966422, 'difficulty_error_LAG_3': 0.0053663825439096, 'guessing_error_LAG_3': 0.1192475660138131, 'inattention_error_LAG_3': 0.09101050283976, 'auc_roc_LAG_3': 0.8389209009952855, 'optimal_threshold_LAG_3': 0.6877192908405967, 'tpr_LAG_3': 0.7831325301204819, 'tnr_LAG_3': 0.7391304347826086, 'student_mean_accuracy_LAG_3': 0.6434108527131783, 'discriminability_LAG_4': 7.268901739514797, 'difficulty_LAG_4': 0.0718098162627701, 'guessing_LAG_4': 1.5112603682245282e-09, 'inattention_LAG_4': 0.999999999999989, 'discriminability_error_LAG_4': 4.738618081564904, 'difficulty_error_LAG_4': 0.070457822968662, 'guessing_error_LAG_4': 0.2179396738923525, 'inattention_error_LAG_4': 0.3241416635759629, 'auc_roc_LAG_4': 0.5948072522443232, 'optimal_threshold_LAG_4': 0.6914708028155719, 'tpr_LAG_4': 0.1821862348178137, 'tnr_LAG_4': 0.9982608695652174, 'student_mean_accuracy_LAG_4': 0.4621141253507951, 'discriminability_LAG_5': 99.99999999999984, 'difficulty_LAG_5': 0.0484811904390762, 'guessing_LAG_5': 5.765890451334314e-16, 'inattention_LAG_5': 1.0, 'discriminability_error_LAG_5': 9.494442785793176, 'difficulty_error_LAG_5': 0.0010886318145058, 'guessing_error_LAG_5': 0.0233845125142748, 'inattention_error_LAG_5': 0.0246445968250694, 'auc_roc_LAG_5': 0.9499025997205478, 'optimal_threshold_LAG_5': 0.466532350160636, 'tpr_LAG_5': 0.8940886699507389, 'tnr_LAG_5': 0.8964803312629399, 'student_mean_accuracy_LAG_5': 0.4566929133858268, 'discriminability_LAG_6': 25.01809724540534, 'difficulty_LAG_6': 0.0089738096745149, 'guessing_LAG_6': 1.4959830151019587e-19, 'inattention_LAG_6': 0.9999999995680684, 'discriminability_error_LAG_6': 4.527072784934737, 'difficulty_error_LAG_6': 0.0077265812702103, 'guessing_error_LAG_6': 0.0717109713609006, 'inattention_error_LAG_6': 0.0428105068996865, 'auc_roc_LAG_6': 0.7855934292501456, 'optimal_threshold_LAG_6': 0.5982390911885075, 'tpr_LAG_6': 0.8258706467661692, 'tnr_LAG_6': 0.5709459459459459, 'student_mean_accuracy_LAG_6': 0.6707452725250278, 'discriminability_LAG_7': 45.50218172642221, 'difficulty_LAG_7': 0.0438525846393088, 'guessing_LAG_7': 4.795570228712377e-15, 'inattention_LAG_7': 0.99999999775728, 'discriminability_error_LAG_7': 6.151254737508041, 'difficulty_error_LAG_7': 0.0031484958480424, 'guessing_error_LAG_7': 0.0446137205799492, 'inattention_error_LAG_7': 0.0370673219013597, 'auc_roc_LAG_7': 0.8515243788877713, 'optimal_threshold_LAG_7': 0.4557291307203294, 'tpr_LAG_7': 0.8195329087048833, 'tnr_LAG_7': 0.7171052631578947, 'student_mean_accuracy_LAG_7': 0.5080906148867314, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 4.828313737302301, 'LOG_sample_size_LAG_1': 4.663439094112067, 'LOG_sample_size_LAG_2': 4.727387818712341, 'LOG_sample_size_LAG_3': 4.859812404361672, 'LOG_sample_size_LAG_4': 6.974478911025045, 'LOG_sample_size_LAG_5': 6.790097235513905, 'LOG_sample_size_LAG_6': 6.80128303447162, 'LOG_sample_size_LAG_7': 6.831953565565855, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': 0.069366838812882, 'optimal_threshold_spread': 0.23574167209524255, 'student_mean_accuracy_spread': 0.214052359139201, 'tnr_spread': 0.4273149236192715, 'tpr_spread': 0.7119024351329252, 'mean_auc_roc': 0.8254329075047403, 'mean_discriminability': 64.13318024854865, 'mean_discriminability_error': 18.77637089289783, 'mean_difficulty_error': 0.013834397756965214, 'mean_guessing_error': 0.09948073417099347, 'mean_inattention_error': 0.10962574843366633, 'num_responses': 7.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 0.0, 'CORRECTNESS_LAG_4': 1.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': nan, 'CORRECTNESS_LAG_7': nan, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': nan, 'DURATIONSECONDS_LAG_2': nan, 'DURATIONSECONDS_LAG_3': nan, 'DURATIONSECONDS_LAG_4': nan, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.032577, 'OCCURREDAT_DIFF_2': 119.564716, 'OCCURREDAT_DIFF_3': 48.232196, 'OCCURREDAT_DIFF_4': 0.005818, 'OCCURREDAT_DIFF_5': 93.699182, 'OCCURREDAT_DIFF_6': nan, 'OCCURREDAT_DIFF_7': nan, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -3.424148761079264, 'LOG_OCCURREDAT_DIFF_2': 4.7838577812681295, 'LOG_OCCURREDAT_DIFF_3': 3.876026764871002, 'LOG_OCCURREDAT_DIFF_4': -5.146798718909127, 'LOG_OCCURREDAT_DIFF_5': 4.540089459216942, 'LOG_OCCURREDAT_DIFF_6': nan, 'LOG_OCCURREDAT_DIFF_7': nan, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': nan, 'LOG_DURATIONSECONDS_LAG_2': nan, 'LOG_DURATIONSECONDS_LAG_3': nan, 'LOG_DURATIONSECONDS_LAG_4': nan, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': nan, 'difficulty': nan, 'guessing': nan, 'inattention': nan, 'discriminability_error': nan, 'difficulty_error': nan, 'guessing_error': nan, 'inattention_error': nan, 'auc_roc': nan, 'optimal_threshold': nan, 'tpr': nan, 'tnr': nan, 'skill_optimal_threshold': nan, 'student_mean_accuracy': nan, 'discriminability_LAG_1': nan, 'difficulty_LAG_1': nan, 'guessing_LAG_1': nan, 'inattention_LAG_1': nan, 'discriminability_error_LAG_1': nan, 'difficulty_error_LAG_1': nan, 'guessing_error_LAG_1': nan, 'inattention_error_LAG_1': nan, 'auc_roc_LAG_1': nan, 'optimal_threshold_LAG_1': nan, 'tpr_LAG_1': nan, 'tnr_LAG_1': nan, 'student_mean_accuracy_LAG_1': nan, 'discriminability_LAG_2': nan, 'difficulty_LAG_2': nan, 'guessing_LAG_2': nan, 'inattention_LAG_2': nan, 'discriminability_error_LAG_2': nan, 'difficulty_error_LAG_2': nan, 'guessing_error_LAG_2': nan, 'inattention_error_LAG_2': nan, 'auc_roc_LAG_2': nan, 'optimal_threshold_LAG_2': nan, 'tpr_LAG_2': nan, 'tnr_LAG_2': nan, 'student_mean_accuracy_LAG_2': nan, 'discriminability_LAG_3': nan, 'difficulty_LAG_3': nan, 'guessing_LAG_3': nan, 'inattention_LAG_3': nan, 'discriminability_error_LAG_3': nan, 'difficulty_error_LAG_3': nan, 'guessing_error_LAG_3': nan, 'inattention_error_LAG_3': nan, 'auc_roc_LAG_3': nan, 'optimal_threshold_LAG_3': nan, 'tpr_LAG_3': nan, 'tnr_LAG_3': nan, 'student_mean_accuracy_LAG_3': nan, 'discriminability_LAG_4': nan, 'difficulty_LAG_4': nan, 'guessing_LAG_4': nan, 'inattention_LAG_4': nan, 'discriminability_error_LAG_4': nan, 'difficulty_error_LAG_4': nan, 'guessing_error_LAG_4': nan, 'inattention_error_LAG_4': nan, 'auc_roc_LAG_4': nan, 'optimal_threshold_LAG_4': nan, 'tpr_LAG_4': nan, 'tnr_LAG_4': nan, 'student_mean_accuracy_LAG_4': nan, 'discriminability_LAG_5': nan, 'difficulty_LAG_5': nan, 'guessing_LAG_5': nan, 'inattention_LAG_5': nan, 'discriminability_error_LAG_5': nan, 'difficulty_error_LAG_5': nan, 'guessing_error_LAG_5': nan, 'inattention_error_LAG_5': nan, 'auc_roc_LAG_5': nan, 'optimal_threshold_LAG_5': nan, 'tpr_LAG_5': nan, 'tnr_LAG_5': nan, 'student_mean_accuracy_LAG_5': nan, 'discriminability_LAG_6': nan, 'difficulty_LAG_6': nan, 'guessing_LAG_6': nan, 'inattention_LAG_6': nan, 'discriminability_error_LAG_6': nan, 'difficulty_error_LAG_6': nan, 'guessing_error_LAG_6': nan, 'inattention_error_LAG_6': nan, 'auc_roc_LAG_6': nan, 'optimal_threshold_LAG_6': nan, 'tpr_LAG_6': nan, 'tnr_LAG_6': nan, 'student_mean_accuracy_LAG_6': nan, 'discriminability_LAG_7': nan, 'difficulty_LAG_7': nan, 'guessing_LAG_7': nan, 'inattention_LAG_7': nan, 'discriminability_error_LAG_7': nan, 'difficulty_error_LAG_7': nan, 'guessing_error_LAG_7': nan, 'inattention_error_LAG_7': nan, 'auc_roc_LAG_7': nan, 'optimal_threshold_LAG_7': nan, 'tpr_LAG_7': nan, 'tnr_LAG_7': nan, 'student_mean_accuracy_LAG_7': nan, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': nan, 'LOG_sample_size_LAG_1': nan, 'LOG_sample_size_LAG_2': nan, 'LOG_sample_size_LAG_3': nan, 'LOG_sample_size_LAG_4': nan, 'LOG_sample_size_LAG_5': nan, 'LOG_sample_size_LAG_6': nan, 'LOG_sample_size_LAG_7': nan, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': nan, 'optimal_threshold_spread': nan, 'student_mean_accuracy_spread': nan, 'tnr_spread': nan, 'tpr_spread': nan, 'mean_auc_roc': nan, 'mean_discriminability': nan, 'mean_discriminability_error': nan, 'mean_difficulty_error': nan, 'mean_guessing_error': nan, 'mean_inattention_error': nan, 'num_responses': 5.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 1.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 1.0, 'CORRECTNESS_LAG_7': 1.0, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': nan, 'DURATIONSECONDS_LAG_2': nan, 'DURATIONSECONDS_LAG_3': nan, 'DURATIONSECONDS_LAG_4': nan, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.001189, 'OCCURREDAT_DIFF_2': 0.001237, 'OCCURREDAT_DIFF_3': 0.001121, 'OCCURREDAT_DIFF_4': 0.00106, 'OCCURREDAT_DIFF_5': 0.000982, 'OCCURREDAT_DIFF_6': 0.001222, 'OCCURREDAT_DIFF_7': 0.001348, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -6.734642661273492, 'LOG_OCCURREDAT_DIFF_2': -6.695066185571786, 'LOG_OCCURREDAT_DIFF_3': -6.793534134892114, 'LOG_OCCURREDAT_DIFF_4': -6.849486370858161, 'LOG_OCCURREDAT_DIFF_5': -6.925919249609808, 'LOG_OCCURREDAT_DIFF_6': -6.7072664182327335, 'LOG_OCCURREDAT_DIFF_7': -6.609133266492021, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': nan, 'LOG_DURATIONSECONDS_LAG_2': nan, 'LOG_DURATIONSECONDS_LAG_3': nan, 'LOG_DURATIONSECONDS_LAG_4': nan, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': nan, 'difficulty': nan, 'guessing': nan, 'inattention': nan, 'discriminability_error': nan, 'difficulty_error': nan, 'guessing_error': nan, 'inattention_error': nan, 'auc_roc': nan, 'optimal_threshold': nan, 'tpr': nan, 'tnr': nan, 'skill_optimal_threshold': nan, 'student_mean_accuracy': nan, 'discriminability_LAG_1': nan, 'difficulty_LAG_1': nan, 'guessing_LAG_1': nan, 'inattention_LAG_1': nan, 'discriminability_error_LAG_1': nan, 'difficulty_error_LAG_1': nan, 'guessing_error_LAG_1': nan, 'inattention_error_LAG_1': nan, 'auc_roc_LAG_1': nan, 'optimal_threshold_LAG_1': nan, 'tpr_LAG_1': nan, 'tnr_LAG_1': nan, 'student_mean_accuracy_LAG_1': nan, 'discriminability_LAG_2': nan, 'difficulty_LAG_2': nan, 'guessing_LAG_2': nan, 'inattention_LAG_2': nan, 'discriminability_error_LAG_2': nan, 'difficulty_error_LAG_2': nan, 'guessing_error_LAG_2': nan, 'inattention_error_LAG_2': nan, 'auc_roc_LAG_2': nan, 'optimal_threshold_LAG_2': nan, 'tpr_LAG_2': nan, 'tnr_LAG_2': nan, 'student_mean_accuracy_LAG_2': nan, 'discriminability_LAG_3': nan, 'difficulty_LAG_3': nan, 'guessing_LAG_3': nan, 'inattention_LAG_3': nan, 'discriminability_error_LAG_3': nan, 'difficulty_error_LAG_3': nan, 'guessing_error_LAG_3': nan, 'inattention_error_LAG_3': nan, 'auc_roc_LAG_3': nan, 'optimal_threshold_LAG_3': nan, 'tpr_LAG_3': nan, 'tnr_LAG_3': nan, 'student_mean_accuracy_LAG_3': nan, 'discriminability_LAG_4': nan, 'difficulty_LAG_4': nan, 'guessing_LAG_4': nan, 'inattention_LAG_4': nan, 'discriminability_error_LAG_4': nan, 'difficulty_error_LAG_4': nan, 'guessing_error_LAG_4': nan, 'inattention_error_LAG_4': nan, 'auc_roc_LAG_4': nan, 'optimal_threshold_LAG_4': nan, 'tpr_LAG_4': nan, 'tnr_LAG_4': nan, 'student_mean_accuracy_LAG_4': nan, 'discriminability_LAG_5': nan, 'difficulty_LAG_5': nan, 'guessing_LAG_5': nan, 'inattention_LAG_5': nan, 'discriminability_error_LAG_5': nan, 'difficulty_error_LAG_5': nan, 'guessing_error_LAG_5': nan, 'inattention_error_LAG_5': nan, 'auc_roc_LAG_5': nan, 'optimal_threshold_LAG_5': nan, 'tpr_LAG_5': nan, 'tnr_LAG_5': nan, 'student_mean_accuracy_LAG_5': nan, 'discriminability_LAG_6': nan, 'difficulty_LAG_6': nan, 'guessing_LAG_6': nan, 'inattention_LAG_6': nan, 'discriminability_error_LAG_6': nan, 'difficulty_error_LAG_6': nan, 'guessing_error_LAG_6': nan, 'inattention_error_LAG_6': nan, 'auc_roc_LAG_6': nan, 'optimal_threshold_LAG_6': nan, 'tpr_LAG_6': nan, 'tnr_LAG_6': nan, 'student_mean_accuracy_LAG_6': nan, 'discriminability_LAG_7': nan, 'difficulty_LAG_7': nan, 'guessing_LAG_7': nan, 'inattention_LAG_7': nan, 'discriminability_error_LAG_7': nan, 'difficulty_error_LAG_7': nan, 'guessing_error_LAG_7': nan, 'inattention_error_LAG_7': nan, 'auc_roc_LAG_7': nan, 'optimal_threshold_LAG_7': nan, 'tpr_LAG_7': nan, 'tnr_LAG_7': nan, 'student_mean_accuracy_LAG_7': nan, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': nan, 'LOG_sample_size_LAG_1': nan, 'LOG_sample_size_LAG_2': nan, 'LOG_sample_size_LAG_3': nan, 'LOG_sample_size_LAG_4': nan, 'LOG_sample_size_LAG_5': nan, 'LOG_sample_size_LAG_6': nan, 'LOG_sample_size_LAG_7': nan, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': nan, 'optimal_threshold_spread': nan, 'student_mean_accuracy_spread': nan, 'tnr_spread': nan, 'tpr_spread': nan, 'mean_auc_roc': nan, 'mean_discriminability': nan, 'mean_discriminability_error': nan, 'mean_difficulty_error': nan, 'mean_guessing_error': nan, 'mean_inattention_error': nan, 'num_responses': 7.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 1.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 1.0, 'CORRECTNESS_LAG_7': 1.0, 'CORRECTNESS_LAG_8': 1.0, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 6.0, 'DURATIONSECONDS_LAG_2': 6.0, 'DURATIONSECONDS_LAG_3': 6.0, 'DURATIONSECONDS_LAG_4': 5.0, 'DURATIONSECONDS_LAG_5': 50.0, 'DURATIONSECONDS_LAG_6': 9.0, 'DURATIONSECONDS_LAG_7': 15.0, 'DURATIONSECONDS_LAG_8': 21.0, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.004, 'OCCURREDAT_DIFF_2': 0.002, 'OCCURREDAT_DIFF_3': 0.002, 'OCCURREDAT_DIFF_4': 98.435, 'OCCURREDAT_DIFF_5': 0.002, 'OCCURREDAT_DIFF_6': 0.015, 'OCCURREDAT_DIFF_7': 0.003, 'OCCURREDAT_DIFF_8': 0.005, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -5.521460917862246, 'LOG_OCCURREDAT_DIFF_2': -6.214608098422191, 'LOG_OCCURREDAT_DIFF_3': -6.214608098422191, 'LOG_OCCURREDAT_DIFF_4': 4.589396431872051, 'LOG_OCCURREDAT_DIFF_5': -6.214608098422191, 'LOG_OCCURREDAT_DIFF_6': -4.199705077879927, 'LOG_OCCURREDAT_DIFF_7': -5.809142990314027, 'LOG_OCCURREDAT_DIFF_8': -5.298317366548036, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 1.791759469228055, 'LOG_DURATIONSECONDS_LAG_2': 1.791759469228055, 'LOG_DURATIONSECONDS_LAG_3': 1.791759469228055, 'LOG_DURATIONSECONDS_LAG_4': 1.6094379124341003, 'LOG_DURATIONSECONDS_LAG_5': 3.912023005428146, 'LOG_DURATIONSECONDS_LAG_6': 2.197224577336219, 'LOG_DURATIONSECONDS_LAG_7': 2.70805020110221, 'LOG_DURATIONSECONDS_LAG_8': 3.044522437723423, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 99.99999999999996, 'difficulty': nan, 'guessing': 1.0215667916445709e-13, 'inattention': 1.0, 'discriminability_error': 2.869513867506209, 'difficulty_error': 0.0003322288605783, 'guessing_error': 0.0072907875295859, 'inattention_error': 0.005043786401199, 'auc_roc': 0.9615831954219578, 'optimal_threshold': 0.5782325053154133, 'tpr': 0.8455993930197269, 'tnr': 0.9518508930882734, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.5771209633278599, 'discriminability_LAG_1': 99.9999999997516, 'difficulty_LAG_1': nan, 'guessing_LAG_1': 4.561337932656272e-14, 'inattention_LAG_1': 0.999999999999995, 'discriminability_error_LAG_1': 2.9618752145499663, 'difficulty_error_LAG_1': 0.000346929728396, 'guessing_error_LAG_1': 0.0084805226637373, 'inattention_error_LAG_1': 0.0047464858213457, 'auc_roc_LAG_1': 0.959160390369728, 'optimal_threshold_LAG_1': 0.7263038355655347, 'tpr_LAG_1': 0.8096137339055794, 'tnr_LAG_1': 0.9565061107117182, 'student_mean_accuracy_LAG_1': 0.6767747182525851, 'discriminability_LAG_2': 15.343867057296547, 'difficulty_LAG_2': nan, 'guessing_LAG_2': 3.6186192503188246e-09, 'inattention_LAG_2': 0.9999999999997125, 'discriminability_error_LAG_2': 0.8482213126971729, 'difficulty_error_LAG_2': 0.0033855480193511, 'guessing_error_LAG_2': 0.0197859958390759, 'inattention_error_LAG_2': 0.0114630648928105, 'auc_roc_LAG_2': 0.8127324969871224, 'optimal_threshold_LAG_2': 0.6377302527777674, 'tpr_LAG_2': 0.5200729927007299, 'tnr_LAG_2': 0.9636071715279636, 'student_mean_accuracy_LAG_2': 0.5398349956901859, 'discriminability_LAG_3': 3.447123366764894, 'difficulty_LAG_3': nan, 'guessing_LAG_3': 2.4180644980577646e-10, 'inattention_LAG_3': 1.0, 'discriminability_error_LAG_3': 0.8910641623774233, 'difficulty_error_LAG_3': 0.0687096311219514, 'guessing_error_LAG_3': 0.1427286928166691, 'inattention_error_LAG_3': 0.0625289044761049, 'auc_roc_LAG_3': 0.6588256041377887, 'optimal_threshold_LAG_3': 0.5481732237674467, 'tpr_LAG_3': 0.4074952561669829, 'tnr_LAG_3': 0.8297872340425532, 'student_mean_accuracy_LAG_3': 0.5547368421052632, 'discriminability_LAG_4': 9.710750548656938, 'difficulty_LAG_4': nan, 'guessing_LAG_4': 2.1904123526475107e-11, 'inattention_LAG_4': 0.9999999999999774, 'discriminability_error_LAG_4': 0.6642726737615613, 'difficulty_error_LAG_4': 0.0082620561146286, 'guessing_error_LAG_4': 0.0352169442170615, 'inattention_error_LAG_4': 0.0121131630837305, 'auc_roc_LAG_4': 0.7588606874598047, 'optimal_threshold_LAG_4': 0.663413875508571, 'tpr_LAG_4': 0.4914148351648351, 'tnr_LAG_4': 0.8769083969465649, 'student_mean_accuracy_LAG_4': 0.6494201605709188, 'discriminability_LAG_5': 12.51279255966097, 'difficulty_LAG_5': nan, 'guessing_LAG_5': 3.1810283203947745e-11, 'inattention_LAG_5': 0.9999999999979048, 'discriminability_error_LAG_5': 0.7159668009276678, 'difficulty_error_LAG_5': 0.0045448854604382, 'guessing_error_LAG_5': 0.023380696029044, 'inattention_error_LAG_5': 0.0107346372334499, 'auc_roc_LAG_5': 0.7917504537955176, 'optimal_threshold_LAG_5': 0.6824333198514945, 'tpr_LAG_5': 0.4756727855997148, 'tnr_LAG_5': 0.9584450402144772, 'student_mean_accuracy_LAG_5': 0.6006851514827106, 'discriminability_LAG_6': 9.14845068415605, 'difficulty_LAG_6': nan, 'guessing_LAG_6': 3.0813071109360505e-12, 'inattention_LAG_6': 0.9999999999053196, 'discriminability_error_LAG_6': 0.6878962180389158, 'difficulty_error_LAG_6': 0.0078573000018755, 'guessing_error_LAG_6': 0.0306805594604167, 'inattention_error_LAG_6': 0.0162654372132923, 'auc_roc_LAG_6': 0.7741225807935903, 'optimal_threshold_LAG_6': 0.6263084516593317, 'tpr_LAG_6': 0.4297985781990521, 'tnr_LAG_6': 0.9486376448481052, 'student_mean_accuracy_LAG_6': 0.5139290607398387, 'discriminability_LAG_7': 2.9333285316809024, 'difficulty_LAG_7': nan, 'guessing_LAG_7': 2.333384501007695e-10, 'inattention_LAG_7': 1.0, 'discriminability_error_LAG_7': 0.8402280810322483, 'difficulty_error_LAG_7': 0.0922683052863515, 'guessing_error_LAG_7': 0.1720300972162351, 'inattention_error_LAG_7': 0.0713384951104603, 'auc_roc_LAG_7': 0.6335280332388006, 'optimal_threshold_LAG_7': 0.5692626581022144, 'tpr_LAG_7': 0.356516290726817, 'tnr_LAG_7': 0.8355537052456287, 'student_mean_accuracy_LAG_7': 0.570611369324276, 'discriminability_LAG_8': 3.447123366764894, 'difficulty_LAG_8': nan, 'guessing_LAG_8': 2.4180644980577646e-10, 'inattention_LAG_8': 1.0, 'discriminability_error_LAG_8': 0.8910641623774233, 'difficulty_error_LAG_8': 0.0687096311219514, 'guessing_error_LAG_8': 0.1427286928166691, 'inattention_error_LAG_8': 0.0625289044761049, 'auc_roc_LAG_8': 0.6588256041377887, 'optimal_threshold_LAG_8': 0.5481732237674467, 'tpr_LAG_8': 0.4074952561669829, 'tnr_LAG_8': 0.8297872340425532, 'student_mean_accuracy_LAG_8': 0.5547368421052632, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 9.119868468812108, 'LOG_sample_size_LAG_1': 9.060331104649475, 'LOG_sample_size_LAG_2': 9.00220857828241, 'LOG_sample_size_LAG_3': 8.242756345714477, 'LOG_sample_size_LAG_4': 9.101417964751995, 'LOG_sample_size_LAG_5': 9.14216859187285, 'LOG_sample_size_LAG_6': 8.790116892892472, 'LOG_sample_size_LAG_7': 8.629449873761905, 'LOG_sample_size_LAG_8': 8.242756345714477, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': nan, 'optimal_threshold_spread': 0.17813061179808798, 'student_mean_accuracy_spread': 0.16284565751274638, 'tnr_spread': 0.13381993748541043, 'tpr_spread': 0.45309744317876244, 'mean_auc_roc': 0.7559757313650176, 'mean_discriminability': 19.5679295143416, 'mean_discriminability_error': 1.0625735782202974, 'mean_difficulty_error': 0.031760535856867965, 'mean_guessing_error': 0.07187902513236359, 'mean_inattention_error': 0.031464886538412375, 'num_responses': 8.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 1.0, 'CORRECTNESS_LAG_5': 0.0, 'CORRECTNESS_LAG_6': 0.0, 'CORRECTNESS_LAG_7': 1.0, 'CORRECTNESS_LAG_8': 1.0, 'CORRECTNESS_LAG_9': 1.0, 'CORRECTNESS_LAG_10': 0.0, 'DURATIONSECONDS_LAG_1': nan, 'DURATIONSECONDS_LAG_2': nan, 'DURATIONSECONDS_LAG_3': nan, 'DURATIONSECONDS_LAG_4': nan, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 503.891, 'OCCURREDAT_DIFF_2': 0.001, 'OCCURREDAT_DIFF_3': 0.001, 'OCCURREDAT_DIFF_4': 0.001, 'OCCURREDAT_DIFF_5': 0.003, 'OCCURREDAT_DIFF_6': 0.005, 'OCCURREDAT_DIFF_7': 0.002, 'OCCURREDAT_DIFF_8': 0.002, 'OCCURREDAT_DIFF_9': 0.001, 'OCCURREDAT_DIFF_10': 0.026, 'LOG_OCCURREDAT_DIFF_1': 6.222359974840404, 'LOG_OCCURREDAT_DIFF_2': -6.907755278982137, 'LOG_OCCURREDAT_DIFF_3': -6.907755278982137, 'LOG_OCCURREDAT_DIFF_4': -6.907755278982137, 'LOG_OCCURREDAT_DIFF_5': -5.809142990314027, 'LOG_OCCURREDAT_DIFF_6': -5.298317366548036, 'LOG_OCCURREDAT_DIFF_7': -6.214608098422191, 'LOG_OCCURREDAT_DIFF_8': -6.214608098422191, 'LOG_OCCURREDAT_DIFF_9': -6.907755278982137, 'LOG_OCCURREDAT_DIFF_10': -3.649658740960655, 'LOG_DURATIONSECONDS_LAG_1': nan, 'LOG_DURATIONSECONDS_LAG_2': nan, 'LOG_DURATIONSECONDS_LAG_3': nan, 'LOG_DURATIONSECONDS_LAG_4': nan, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 100.0, 'difficulty': nan, 'guessing': 0.0970199731475763, 'inattention': 1.0, 'discriminability_error': 16.230651833998134, 'difficulty_error': 0.0017443740951758, 'guessing_error': 0.0255313076755434, 'inattention_error': 0.0383047128804775, 'auc_roc': 0.8888832040603319, 'optimal_threshold': 0.5262217334269678, 'tpr': 0.7022900763358778, 'tnr': 0.9436997319034852, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.4125984251968504, 'discriminability_LAG_1': 95.030903643694, 'difficulty_LAG_1': nan, 'guessing_LAG_1': 8.760338471784794e-11, 'inattention_LAG_1': 0.9957929745463416, 'discriminability_error_LAG_1': 11.419118214361784, 'difficulty_error_LAG_1': 0.0023832905262054, 'guessing_error_LAG_1': 0.0852888895994428, 'inattention_error_LAG_1': 0.0092736378472533, 'auc_roc_LAG_1': 0.9213531258920924, 'optimal_threshold_LAG_1': 0.9274487210018514, 'tpr_LAG_1': 0.8571428571428571, 'tnr_LAG_1': 0.8709677419354839, 'student_mean_accuracy_LAG_1': 0.9273153575615476, 'discriminability_LAG_2': 99.99999999997108, 'difficulty_LAG_2': nan, 'guessing_LAG_2': 4.924840243864334e-11, 'inattention_LAG_2': 0.9999999999999992, 'discriminability_error_LAG_2': 9.560794179032744, 'difficulty_error_LAG_2': 0.0013499992084602, 'guessing_error_LAG_2': 0.0401916120871034, 'inattention_error_LAG_2': 0.0083681916711846, 'auc_roc_LAG_2': 0.9569713820697138, 'optimal_threshold_LAG_2': 0.9373221136832808, 'tpr_LAG_2': 0.8477722772277227, 'tnr_LAG_2': 0.9589041095890412, 'student_mean_accuracy_LAG_2': 0.9171396140749148, 'discriminability_LAG_3': 100.0, 'difficulty_LAG_3': nan, 'guessing_LAG_3': 0.0568906511886302, 'inattention_LAG_3': 0.9999999999980692, 'discriminability_error_LAG_3': 11.88388626054641, 'difficulty_error_LAG_3': 0.0019097710390611, 'guessing_error_LAG_3': 0.0645724765528287, 'inattention_error_LAG_3': 0.0109388343471617, 'auc_roc_LAG_3': 0.9124166543191584, 'optimal_threshold_LAG_3': 0.8972062019898651, 'tpr_LAG_3': 0.8198992443324937, 'tnr_LAG_3': 0.8823529411764706, 'student_mean_accuracy_LAG_3': 0.9032992036405004, 'discriminability_LAG_4': 100.0, 'difficulty_LAG_4': nan, 'guessing_LAG_4': 2.21535690630714e-20, 'inattention_LAG_4': 1.0, 'discriminability_error_LAG_4': 10.604585253923744, 'difficulty_error_LAG_4': 0.001273861548037, 'guessing_error_LAG_4': 0.0394926259939771, 'inattention_error_LAG_4': 0.0177950487371679, 'auc_roc_LAG_4': 0.9461284513805522, 'optimal_threshold_LAG_4': 0.7455467677540261, 'tpr_LAG_4': 0.8037676609105181, 'tnr_LAG_4': 0.9301470588235294, 'student_mean_accuracy_LAG_4': 0.7007700770077008, 'discriminability_LAG_5': 99.99999999992544, 'difficulty_LAG_5': nan, 'guessing_LAG_5': 0.0911110932445534, 'inattention_LAG_5': 1.0, 'discriminability_error_LAG_5': 12.416793646115872, 'difficulty_error_LAG_5': 0.0014062808028734, 'guessing_error_LAG_5': 0.0340899834702629, 'inattention_error_LAG_5': 0.0222104609726894, 'auc_roc_LAG_5': 0.926451288638204, 'optimal_threshold_LAG_5': 0.7060591870527986, 'tpr_LAG_5': 0.800658978583196, 'tnr_LAG_5': 0.9423728813559322, 'student_mean_accuracy_LAG_5': 0.6729490022172949, 'discriminability_LAG_6': 37.15553395458107, 'difficulty_LAG_6': nan, 'guessing_LAG_6': 0.0891587514688244, 'inattention_LAG_6': 0.9999999999999998, 'discriminability_error_LAG_6': 7.490275486972342, 'difficulty_error_LAG_6': 0.0063530351551761, 'guessing_error_LAG_6': 0.0807239430208862, 'inattention_error_LAG_6': 0.0586311865329651, 'auc_roc_LAG_6': 0.7832235457410379, 'optimal_threshold_LAG_6': 0.6079276330758999, 'tpr_LAG_6': 0.6566265060240963, 'tnr_LAG_6': 0.7818696883852692, 'student_mean_accuracy_LAG_6': 0.5851938895417156, 'discriminability_LAG_7': 54.577098301214136, 'difficulty_LAG_7': nan, 'guessing_LAG_7': 0.1268083278093527, 'inattention_LAG_7': 0.9999999999999344, 'discriminability_error_LAG_7': 9.80234764171272, 'difficulty_error_LAG_7': 0.0035985331198601, 'guessing_error_LAG_7': 0.0502012900628231, 'inattention_error_LAG_7': 0.048172182892639, 'auc_roc_LAG_7': 0.8153050108932463, 'optimal_threshold_LAG_7': 0.503999040132022, 'tpr_LAG_7': 0.7155555555555555, 'tnr_LAG_7': 0.7696078431372549, 'student_mean_accuracy_LAG_7': 0.5244755244755245, 'discriminability_LAG_8': 22.33728938435843, 'difficulty_LAG_8': nan, 'guessing_LAG_8': 1.9616879525085148e-12, 'inattention_LAG_8': 1.0, 'discriminability_error_LAG_8': 5.245025241734681, 'difficulty_error_LAG_8': 0.0114648228549023, 'guessing_error_LAG_8': 0.0936857913943714, 'inattention_error_LAG_8': 0.0859581306091741, 'auc_roc_LAG_8': 0.7543363665812646, 'optimal_threshold_LAG_8': 0.3518193528141468, 'tpr_LAG_8': 0.7597402597402597, 'tnr_LAG_8': 0.6122448979591837, 'student_mean_accuracy_LAG_8': 0.411214953271028, 'discriminability_LAG_9': 9.540688245795083, 'difficulty_LAG_9': nan, 'guessing_LAG_9': 2.9008218115600207e-16, 'inattention_LAG_9': 0.9999999999964349, 'discriminability_error_LAG_9': 4.41725846372496, 'difficulty_error_LAG_9': 0.0477432864627668, 'guessing_error_LAG_9': 0.1825980721918247, 'inattention_error_LAG_9': 0.1725963959346704, 'auc_roc_LAG_9': 0.691227038277079, 'optimal_threshold_LAG_9': 0.3332343583645137, 'tpr_LAG_9': 0.8081395348837209, 'tnr_LAG_9': 0.5058365758754864, 'student_mean_accuracy_LAG_9': 0.4009324009324009, 'discriminability_LAG_10': 99.99999999991569, 'difficulty_LAG_10': nan, 'guessing_LAG_10': 0.3101398748202676, 'inattention_LAG_10': 1.0, 'discriminability_error_LAG_10': 41.84237360499269, 'difficulty_error_LAG_10': 0.0048377515689029, 'guessing_error_LAG_10': 0.0418307352048373, 'inattention_error_LAG_10': 0.0837877752424242, 'auc_roc_LAG_10': 0.75004860976084, 'optimal_threshold_LAG_10': 0.6670830310367963, 'tpr_LAG_10': 0.4172661870503597, 'tnr_LAG_10': 0.9864864864864864, 'student_mean_accuracy_LAG_10': 0.4843205574912892, 'LOG_sample_size': 6.453624998892692, 'LOG_sample_size_LAG_1': 6.748759547491679, 'LOG_sample_size_LAG_2': 6.781057625936179, 'LOG_sample_size_LAG_3': 6.778784897685177, 'LOG_sample_size_LAG_4': 6.812345094177479, 'LOG_sample_size_LAG_5': 6.804614520062623, 'LOG_sample_size_LAG_6': 6.746412128573374, 'LOG_sample_size_LAG_7': 6.754604099487962, 'LOG_sample_size_LAG_8': 6.618738983517219, 'LOG_sample_size_LAG_9': 6.061456918928017, 'LOG_sample_size_LAG_10': 5.6594822157596205, 'difficulty_spread': nan, 'optimal_threshold_spread': 0.6040877553187671, 'student_mean_accuracy_spread': 0.5263829566291467, 'tnr_spread': 0.48064991061100004, 'tpr_spread': 0.43987667009249737, 'mean_auc_roc': 0.8457461473553188, 'mean_discriminability': 71.86415135294548, 'mean_discriminability_error': 12.468245799311795, 'mean_difficulty_error': 0.00823206322862453, 'mean_guessing_error': 0.07126754195783576, 'mean_inattention_error': 0.05177318447873297, 'num_responses': 10.0}"
log_text = "{'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': nan, 'CORRECTNESS_LAG_4': nan, 'CORRECTNESS_LAG_5': nan, 'CORRECTNESS_LAG_6': nan, 'CORRECTNESS_LAG_7': nan, 'CORRECTNESS_LAG_8': nan, 'CORRECTNESS_LAG_9': nan, 'CORRECTNESS_LAG_10': nan, 'DURATIONSECONDS_LAG_1': 5.0, 'DURATIONSECONDS_LAG_2': 4.0, 'DURATIONSECONDS_LAG_3': nan, 'DURATIONSECONDS_LAG_4': nan, 'DURATIONSECONDS_LAG_5': nan, 'DURATIONSECONDS_LAG_6': nan, 'DURATIONSECONDS_LAG_7': nan, 'DURATIONSECONDS_LAG_8': nan, 'DURATIONSECONDS_LAG_9': nan, 'DURATIONSECONDS_LAG_10': nan, 'OCCURREDAT_DIFF_1': 0.0002777777777777778, 'OCCURREDAT_DIFF_2': 0.0002777777777777778, 'OCCURREDAT_DIFF_3': nan, 'OCCURREDAT_DIFF_4': nan, 'OCCURREDAT_DIFF_5': nan, 'OCCURREDAT_DIFF_6': nan, 'OCCURREDAT_DIFF_7': nan, 'OCCURREDAT_DIFF_8': nan, 'OCCURREDAT_DIFF_9': nan, 'OCCURREDAT_DIFF_10': nan, 'LOG_OCCURREDAT_DIFF_1': -8.1886891244442, 'LOG_OCCURREDAT_DIFF_2': -8.1886891244442, 'LOG_OCCURREDAT_DIFF_3': nan, 'LOG_OCCURREDAT_DIFF_4': nan, 'LOG_OCCURREDAT_DIFF_5': nan, 'LOG_OCCURREDAT_DIFF_6': nan, 'LOG_OCCURREDAT_DIFF_7': nan, 'LOG_OCCURREDAT_DIFF_8': nan, 'LOG_OCCURREDAT_DIFF_9': nan, 'LOG_OCCURREDAT_DIFF_10': nan, 'LOG_DURATIONSECONDS_LAG_1': 1.6094379124341003, 'LOG_DURATIONSECONDS_LAG_2': 1.3862943611198906, 'LOG_DURATIONSECONDS_LAG_3': nan, 'LOG_DURATIONSECONDS_LAG_4': nan, 'LOG_DURATIONSECONDS_LAG_5': nan, 'LOG_DURATIONSECONDS_LAG_6': nan, 'LOG_DURATIONSECONDS_LAG_7': nan, 'LOG_DURATIONSECONDS_LAG_8': nan, 'LOG_DURATIONSECONDS_LAG_9': nan, 'LOG_DURATIONSECONDS_LAG_10': nan, 'discriminability': 99.99999999920112, 'difficulty': 0.1073201495756432, 'guessing': 1.3283347553803204e-14, 'inattention': 0.9999999950121438, 'discriminability_error': 3.309356047897224, 'difficulty_error': 0.0007026238687415, 'guessing_error': 0.0267881805034962, 'inattention_error': 0.0022625319845933, 'auc_roc': 0.9348674501392478, 'optimal_threshold': 0.9107397177587394, 'tpr': 0.8857031020647813, 'tnr': 0.8344594594594594, 'skill_optimal_threshold': nan, 'student_mean_accuracy': 0.94524095828323, 'discriminability_LAG_1': 64.67977091774577, 'difficulty_LAG_1': 0.1126968371513973, 'guessing_LAG_1': 7.572151786722e-10, 'inattention_LAG_1': 0.9974326411463584, 'discriminability_error_LAG_1': 2.9215021528987646, 'difficulty_error_LAG_1': 0.0016420163241878, 'guessing_error_LAG_1': 0.0465919347140282, 'inattention_error_LAG_1': 0.0040337087207491, 'auc_roc_LAG_1': 0.8763129804640085, 'optimal_threshold_LAG_1': 0.8223762982936269, 'tpr_LAG_1': 0.8347236704900939, 'tnr_LAG_1': 0.72397476340694, 'student_mean_accuracy_LAG_1': 0.8832197458095413, 'discriminability_LAG_2': 83.91868363268537, 'difficulty_LAG_2': 0.1054788186615149, 'guessing_LAG_2': 2.193610665212572e-13, 'inattention_LAG_2': 0.9988156920132532, 'discriminability_error_LAG_2': 3.094562384669184, 'difficulty_error_LAG_2': 0.0009483623214557, 'guessing_error_LAG_2': 0.0313666891970512, 'inattention_error_LAG_2': 0.0027468459436404, 'auc_roc_LAG_2': 0.907644127763543, 'optimal_threshold_LAG_2': 0.9188587900878522, 'tpr_LAG_2': 0.8335951305713725, 'tnr_LAG_2': 0.8016528925619835, 'student_mean_accuracy_LAG_2': 0.933467741935484, 'discriminability_LAG_3': nan, 'difficulty_LAG_3': nan, 'guessing_LAG_3': nan, 'inattention_LAG_3': nan, 'discriminability_error_LAG_3': nan, 'difficulty_error_LAG_3': nan, 'guessing_error_LAG_3': nan, 'inattention_error_LAG_3': nan, 'auc_roc_LAG_3': nan, 'optimal_threshold_LAG_3': nan, 'tpr_LAG_3': nan, 'tnr_LAG_3': nan, 'student_mean_accuracy_LAG_3': nan, 'discriminability_LAG_4': nan, 'difficulty_LAG_4': nan, 'guessing_LAG_4': nan, 'inattention_LAG_4': nan, 'discriminability_error_LAG_4': nan, 'difficulty_error_LAG_4': nan, 'guessing_error_LAG_4': nan, 'inattention_error_LAG_4': nan, 'auc_roc_LAG_4': nan, 'optimal_threshold_LAG_4': nan, 'tpr_LAG_4': nan, 'tnr_LAG_4': nan, 'student_mean_accuracy_LAG_4': nan, 'discriminability_LAG_5': nan, 'difficulty_LAG_5': nan, 'guessing_LAG_5': nan, 'inattention_LAG_5': nan, 'discriminability_error_LAG_5': nan, 'difficulty_error_LAG_5': nan, 'guessing_error_LAG_5': nan, 'inattention_error_LAG_5': nan, 'auc_roc_LAG_5': nan, 'optimal_threshold_LAG_5': nan, 'tpr_LAG_5': nan, 'tnr_LAG_5': nan, 'student_mean_accuracy_LAG_5': nan, 'discriminability_LAG_6': nan, 'difficulty_LAG_6': nan, 'guessing_LAG_6': nan, 'inattention_LAG_6': nan, 'discriminability_error_LAG_6': nan, 'difficulty_error_LAG_6': nan, 'guessing_error_LAG_6': nan, 'inattention_error_LAG_6': nan, 'auc_roc_LAG_6': nan, 'optimal_threshold_LAG_6': nan, 'tpr_LAG_6': nan, 'tnr_LAG_6': nan, 'student_mean_accuracy_LAG_6': nan, 'discriminability_LAG_7': nan, 'difficulty_LAG_7': nan, 'guessing_LAG_7': nan, 'inattention_LAG_7': nan, 'discriminability_error_LAG_7': nan, 'difficulty_error_LAG_7': nan, 'guessing_error_LAG_7': nan, 'inattention_error_LAG_7': nan, 'auc_roc_LAG_7': nan, 'optimal_threshold_LAG_7': nan, 'tpr_LAG_7': nan, 'tnr_LAG_7': nan, 'student_mean_accuracy_LAG_7': nan, 'discriminability_LAG_8': nan, 'difficulty_LAG_8': nan, 'guessing_LAG_8': nan, 'inattention_LAG_8': nan, 'discriminability_error_LAG_8': nan, 'difficulty_error_LAG_8': nan, 'guessing_error_LAG_8': nan, 'inattention_error_LAG_8': nan, 'auc_roc_LAG_8': nan, 'optimal_threshold_LAG_8': nan, 'tpr_LAG_8': nan, 'tnr_LAG_8': nan, 'student_mean_accuracy_LAG_8': nan, 'discriminability_LAG_9': nan, 'difficulty_LAG_9': nan, 'guessing_LAG_9': nan, 'inattention_LAG_9': nan, 'discriminability_error_LAG_9': nan, 'difficulty_error_LAG_9': nan, 'guessing_error_LAG_9': nan, 'inattention_error_LAG_9': nan, 'auc_roc_LAG_9': nan, 'optimal_threshold_LAG_9': nan, 'tpr_LAG_9': nan, 'tnr_LAG_9': nan, 'student_mean_accuracy_LAG_9': nan, 'discriminability_LAG_10': nan, 'difficulty_LAG_10': nan, 'guessing_LAG_10': nan, 'inattention_LAG_10': nan, 'discriminability_error_LAG_10': nan, 'difficulty_error_LAG_10': nan, 'guessing_error_LAG_10': nan, 'inattention_error_LAG_10': nan, 'auc_roc_LAG_10': nan, 'optimal_threshold_LAG_10': nan, 'tpr_LAG_10': nan, 'tnr_LAG_10': nan, 'student_mean_accuracy_LAG_10': nan, 'LOG_sample_size': 9.28831941329277, 'LOG_sample_size_LAG_1': 9.292657414465396, 'LOG_sample_size_LAG_2': 9.297618380083243, 'LOG_sample_size_LAG_3': nan, 'LOG_sample_size_LAG_4': nan, 'LOG_sample_size_LAG_5': nan, 'LOG_sample_size_LAG_6': nan, 'LOG_sample_size_LAG_7': nan, 'LOG_sample_size_LAG_8': nan, 'LOG_sample_size_LAG_9': nan, 'LOG_sample_size_LAG_10': nan, 'difficulty_spread': 0.007218018489882405, 'optimal_threshold_spread': 0.0964824917942253, 'student_mean_accuracy_spread': 0.05024799612594266, 'tnr_spread': 0.07767812915504346, 'tpr_spread': 0.0011285399187214162, 'mean_auc_roc': 0.8919785541137757, 'mean_discriminability': 74.29922727521557, 'mean_discriminability_error': 3.0080322687839742, 'mean_difficulty_error': 0.00129518932282175, 'mean_guessing_error': 0.0389793119555397, 'mean_inattention_error': 0.0033902773321947497, 'num_responses': 2.0}"
#log_text = str({'CORRECTNESS_LAG_1': [1.0], 'CORRECTNESS_LAG_10': [0.0], 'CORRECTNESS_LAG_2': [1.0], 'CORRECTNESS_LAG_3': [1.0], 'CORRECTNESS_LAG_4': [1.0], 'CORRECTNESS_LAG_5': [0.0], 'CORRECTNESS_LAG_6': [0.0], 'CORRECTNESS_LAG_7': [1.0], 'CORRECTNESS_LAG_8': [1.0], 'CORRECTNESS_LAG_9': [1.0], 'DURATIONSECONDS_LAG_1': [nan], 'DURATIONSECONDS_LAG_10': [nan], 'DURATIONSECONDS_LAG_2': [nan], 'DURATIONSECONDS_LAG_3': [nan], 'DURATIONSECONDS_LAG_4': [nan], 'DURATIONSECONDS_LAG_5': [nan], 'DURATIONSECONDS_LAG_6': [nan], 'DURATIONSECONDS_LAG_7': [nan], 'DURATIONSECONDS_LAG_8': [nan], 'DURATIONSECONDS_LAG_9': [nan], 'LOG_DURATIONSECONDS_LAG_1': [nan], 'LOG_DURATIONSECONDS_LAG_10': [nan], 'LOG_DURATIONSECONDS_LAG_2': [nan], 'LOG_DURATIONSECONDS_LAG_3': [nan], 'LOG_DURATIONSECONDS_LAG_4': [nan], 'LOG_DURATIONSECONDS_LAG_5': [nan], 'LOG_DURATIONSECONDS_LAG_6': [nan], 'LOG_DURATIONSECONDS_LAG_7': [nan], 'LOG_DURATIONSECONDS_LAG_8': [nan], 'LOG_DURATIONSECONDS_LAG_9': [nan], 'LOG_OCCURREDAT_DIFF_1': [6.222359974840404], 'LOG_OCCURREDAT_DIFF_10': [-3.649658740960655], 'LOG_OCCURREDAT_DIFF_2': [-6.907755278982137], 'LOG_OCCURREDAT_DIFF_3': [-6.907755278982137], 'LOG_OCCURREDAT_DIFF_4': [-6.907755278982137], 'LOG_OCCURREDAT_DIFF_5': [-5.809142990314028], 'LOG_OCCURREDAT_DIFF_6': [-5.298317366548036], 'LOG_OCCURREDAT_DIFF_7': [-6.214608098422191], 'LOG_OCCURREDAT_DIFF_8': [-6.214608098422191], 'LOG_OCCURREDAT_DIFF_9': [-6.907755278982137], 'LOG_sample_size': [6.453624998892692], 'LOG_sample_size_LAG_1': [6.748759547491679], 'LOG_sample_size_LAG_10': [5.659482215759621], 'LOG_sample_size_LAG_2': [6.781057625936179], 'LOG_sample_size_LAG_3': [6.778784897685177], 'LOG_sample_size_LAG_4': [6.812345094177479], 'LOG_sample_size_LAG_5': [6.804614520062623], 'LOG_sample_size_LAG_6': [6.7464121285733745], 'LOG_sample_size_LAG_7': [6.754604099487962], 'LOG_sample_size_LAG_8': [6.618738983517219], 'LOG_sample_size_LAG_9': [6.061456918928017], 'OCCURREDAT_DIFF_1': [503.891], 'OCCURREDAT_DIFF_10': [0.026], 'OCCURREDAT_DIFF_2': [0.001], 'OCCURREDAT_DIFF_3': [0.001], 'OCCURREDAT_DIFF_4': [0.001], 'OCCURREDAT_DIFF_5': [0.003], 'OCCURREDAT_DIFF_6': [0.005], 'OCCURREDAT_DIFF_7': [0.002], 'OCCURREDAT_DIFF_8': [0.002], 'OCCURREDAT_DIFF_9': [0.001], 'auc_roc': [0.8888832040603319], 'auc_roc_LAG_1': [0.9213531258920924], 'auc_roc_LAG_10': [0.75004860976084], 'auc_roc_LAG_2': [0.9569713820697138], 'auc_roc_LAG_3': [0.9124166543191584], 'auc_roc_LAG_4': [0.9461284513805522], 'auc_roc_LAG_5': [0.926451288638204], 'auc_roc_LAG_6': [0.7832235457410379], 'auc_roc_LAG_7': [0.8153050108932463], 'auc_roc_LAG_8': [0.7543363665812646], 'auc_roc_LAG_9': [0.691227038277079], 'difficulty': [nan], 'difficulty_LAG_1': [nan], 'difficulty_LAG_10': [nan], 'difficulty_LAG_2': [nan], 'difficulty_LAG_3': [nan], 'difficulty_LAG_4': [nan], 'difficulty_LAG_5': [nan], 'difficulty_LAG_6': [nan], 'difficulty_LAG_7': [nan], 'difficulty_LAG_8': [nan], 'difficulty_LAG_9': [nan], 'difficulty_error': [0.0017443740951758], 'difficulty_error_LAG_1': [0.0023832905262054], 'difficulty_error_LAG_10': [0.0048377515689029], 'difficulty_error_LAG_2': [0.0013499992084602], 'difficulty_error_LAG_3': [0.0019097710390611], 'difficulty_error_LAG_4': [0.001273861548037], 'difficulty_error_LAG_5': [0.0014062808028734], 'difficulty_error_LAG_6': [0.0063530351551761], 'difficulty_error_LAG_7': [0.0035985331198601], 'difficulty_error_LAG_8': [0.0114648228549023], 'difficulty_error_LAG_9': [0.0477432864627668], 'difficulty_spread': [nan], 'discriminability': [100.0], 'discriminability_LAG_1': [95.030903643694], 'discriminability_LAG_10': [99.99999999991569], 'discriminability_LAG_2': [99.99999999997108], 'discriminability_LAG_3': [100.0], 'discriminability_LAG_4': [100.0], 'discriminability_LAG_5': [99.99999999992544], 'discriminability_LAG_6': [37.15553395458107], 'discriminability_LAG_7': [54.577098301214136], 'discriminability_LAG_8': [22.33728938435843], 'discriminability_LAG_9': [9.540688245795083], 'discriminability_error': [16.230651833998134], 'discriminability_error_LAG_1': [11.419118214361784], 'discriminability_error_LAG_10': [41.84237360499269], 'discriminability_error_LAG_2': [9.560794179032744], 'discriminability_error_LAG_3': [11.88388626054641], 'discriminability_error_LAG_4': [10.604585253923744], 'discriminability_error_LAG_5': [12.416793646115872], 'discriminability_error_LAG_6': [7.490275486972342], 'discriminability_error_LAG_7': [9.80234764171272], 'discriminability_error_LAG_8': [5.245025241734681], 'discriminability_error_LAG_9': [4.41725846372496], 'guessing': [0.0970199731475763], 'guessing_LAG_1': [8.760338471784794e-11], 'guessing_LAG_10': [0.3101398748202676], 'guessing_LAG_2': [4.924840243864334e-11], 'guessing_LAG_3': [0.0568906511886302], 'guessing_LAG_4': [2.21535690630714e-20], 'guessing_LAG_5': [0.0911110932445534], 'guessing_LAG_6': [0.0891587514688244], 'guessing_LAG_7': [0.1268083278093527], 'guessing_LAG_8': [1.9616879525085148e-12], 'guessing_LAG_9': [2.9008218115600207e-16], 'guessing_error': [0.0255313076755434], 'guessing_error_LAG_1': [0.0852888895994428], 'guessing_error_LAG_10': [0.0418307352048373], 'guessing_error_LAG_2': [0.0401916120871034], 'guessing_error_LAG_3': [0.0645724765528287], 'guessing_error_LAG_4': [0.0394926259939771], 'guessing_error_LAG_5': [0.0340899834702629], 'guessing_error_LAG_6': [0.0807239430208862], 'guessing_error_LAG_7': [0.0502012900628231], 'guessing_error_LAG_8': [0.0936857913943714], 'guessing_error_LAG_9': [0.1825980721918247], 'inattention': [1.0], 'inattention_LAG_1': [0.9957929745463416], 'inattention_LAG_10': [1.0], 'inattention_LAG_2': [0.9999999999999992], 'inattention_LAG_3': [0.9999999999980692], 'inattention_LAG_4': [1.0], 'inattention_LAG_5': [1.0], 'inattention_LAG_6': [0.9999999999999998], 'inattention_LAG_7': [0.9999999999999344], 'inattention_LAG_8': [1.0], 'inattention_LAG_9': [0.9999999999964349], 'inattention_error': [0.0383047128804775], 'inattention_error_LAG_1': [0.0092736378472533], 'inattention_error_LAG_10': [0.0837877752424242], 'inattention_error_LAG_2': [0.0083681916711846], 'inattention_error_LAG_3': [0.0109388343471617], 'inattention_error_LAG_4': [0.0177950487371679], 'inattention_error_LAG_5': [0.0222104609726894], 'inattention_error_LAG_6': [0.0586311865329651], 'inattention_error_LAG_7': [0.048172182892639], 'inattention_error_LAG_8': [0.0859581306091741], 'inattention_error_LAG_9': [0.1725963959346704], 'mean_auc_roc': [0.8457461473553188], 'mean_difficulty_error': [0.00823206322862453], 'mean_discriminability': [71.86415135294548], 'mean_discriminability_error': [12.468245799311795], 'mean_guessing_error': [0.07126754195783576], 'mean_inattention_error': [0.05177318447873297], 'num_responses': [10.0], 'optimal_threshold': [0.5262217334269678], 'optimal_threshold_LAG_1': [0.9274487210018514], 'optimal_threshold_LAG_10': [0.6670830310367963], 'optimal_threshold_LAG_2': [0.9373221136832808], 'optimal_threshold_LAG_3': [0.8972062019898651], 'optimal_threshold_LAG_4': [0.7455467677540261], 'optimal_threshold_LAG_5': [0.7060591870527986], 'optimal_threshold_LAG_6': [0.6079276330758999], 'optimal_threshold_LAG_7': [0.503999040132022], 'optimal_threshold_LAG_8': [0.3518193528141468], 'optimal_threshold_LAG_9': [0.3332343583645137], 'optimal_threshold_spread': [0.6040877553187671], 'skill_optimal_threshold': [nan], 'student_mean_accuracy': [0.4125984251968504], 'student_mean_accuracy_LAG_1': [0.9273153575615476], 'student_mean_accuracy_LAG_10': [0.4843205574912892], 'student_mean_accuracy_LAG_2': [0.9171396140749148], 'student_mean_accuracy_LAG_3': [0.9032992036405004], 'student_mean_accuracy_LAG_4': [0.7007700770077008], 'student_mean_accuracy_LAG_5': [0.6729490022172949], 'student_mean_accuracy_LAG_6': [0.5851938895417156], 'student_mean_accuracy_LAG_7': [0.5244755244755245], 'student_mean_accuracy_LAG_8': [0.411214953271028], 'student_mean_accuracy_LAG_9': [0.4009324009324009], 'student_mean_accuracy_spread': [0.5263829566291467], 'tnr': [0.9436997319034852], 'tnr_LAG_1': [0.8709677419354839], 'tnr_LAG_10': [0.9864864864864864], 'tnr_LAG_2': [0.9589041095890412], 'tnr_LAG_3': [0.8823529411764706], 'tnr_LAG_4': [0.9301470588235294], 'tnr_LAG_5': [0.9423728813559322], 'tnr_LAG_6': [0.7818696883852692], 'tnr_LAG_7': [0.7696078431372549], 'tnr_LAG_8': [0.6122448979591837], 'tnr_LAG_9': [0.5058365758754864], 'tnr_spread': [0.48064991061100004], 'tpr': [0.7022900763358778], 'tpr_LAG_1': [0.8571428571428571], 'tpr_LAG_10': [0.4172661870503597], 'tpr_LAG_2': [0.8477722772277227], 'tpr_LAG_3': [0.8198992443324937], 'tpr_LAG_4': [0.8037676609105181], 'tpr_LAG_5': [0.800658978583196], 'tpr_LAG_6': [0.6566265060240963], 'tpr_LAG_7': [0.7155555555555555], 'tpr_LAG_8': [0.7597402597402597], 'tpr_LAG_9': [0.8081395348837209], 'tpr_spread': [0.43987667009249737]}, 'parsed_json': {'correctnessHistory': [0, 100, 100, 100, 100, 0, 0, 100, 100, 100, 0], 'durationSecondsHistory': [], 'eventTime': '2024-06-05 07:03:58.636000+00:00', 'eventTimesHistory': ['2024-06-05T07:03:58.000Z', '2024-05-15T07:10:32.000Z', '2024-05-15T07:10:27.000Z', '2024-05-15T07:10:23.000Z', '2024-05-15T07:10:19.000Z', '2024-05-15T07:10:08.000Z', '2024-05-15T07:09:49.000Z', '2024-05-15T07:09:40.000Z', '2024-05-15T07:09:34.000Z', '2024-05-15T07:09:31.000Z', '2024-05-15T07:07:57.000Z'], 'questionId': '94046f4d-7f58-4ed6-ab60-ca6edfc300f4', 'questionIdsHistory': ['94046f4d-7f58-4ed6-ab60-ca6edfc300f4', 'fe63aeb7-9d8e-43a5-857b-6ce7d4a38a63', 'f8187d97-6abf-4d5d-8556-eb4416126031', 'e3f28d4d-43f1-475e-9ecf-b961aac28ed7', 'd330d5a4-b374-47f3-a40a-1180518cbe8f', 'cdd3cff8-3fa0-4a0f-a72e-f0272702950d', 'b22aba47-c2a5-4868-8e52-6be05eb7732a', 'af90d47d-7a56-4653-90b1-3791233f7be3', '9dc1166a-f460-41ec-8dbb-44b1f7b8e477', '888949ce-ca06-452d-8813-3275b51212d5', '7363d704-a28e-417b-b6bc-840864f791ef'], 'skillId': 'evnldqti'})


log_text = log_text.replace('nan', 'float("nan")')
log_dict = eval(log_text)

import numpy as np
if debugInputs:
    for column in processed_inputs.columns:
        if column in log_dict:
            log_val = log_dict[column]
            df_val = processed_inputs[column].iloc[0]

            # Handle nan values first
            if isinstance(log_val, float) and isinstance(df_val, float):
                if np.isnan(log_val) and np.isnan(df_val):
                    continue  # Skip if both are nan
                elif np.isnan(log_val) or np.isnan(df_val):  # Only one is nan
                    print(f"Mismatch in {column}:")
                    print(f"Log value: {log_val}")
                    print(f"DataFrame value: {df_val}")
                    print()
                elif not np.isclose(log_val, df_val, rtol=1e-10):
                    print(f"Mismatch in {column}:")
                    print(f"Log value: {log_val}")
                    print(f"DataFrame value: {df_val}")
                    print()
            elif log_val != df_val:
                print(f"Mismatch in {column}:")
                print(f"Log value: {log_val}")
                print(f"DataFrame value: {df_val}")
                print()
                
print(f'{len(processed_inputs.columns)} columns checked.')

217 columns checked.


In [25]:
timeAnalysis = False

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import copy

def generate_time_series_predictions(base_input, start_time_str, duration_minutes=5):
    # Parse the start time
    start_time = datetime.strptime(start_time_str, "%Y-%m-%dT%H:%M:%S.%fZ")
    
    # Generate time points (every second for 5 minutes)
    time_points = [start_time + timedelta(seconds=i) for i in range(duration_minutes * 60)]
    predictions = []
    
    # Run predictions for each time point
    for time_point in time_points:
        # Create a copy of the input
        modified_input = copy.deepcopy(base_input)
        # Update the eventTime
        modified_input['eventTimesHistory'][0] = time_point.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
        
        # Run inference
        if debugInputs:
            results, _ = run_inference(modified_input)
        else:
            results = run_inference(modified_input)
        predictions.append(results['item_prediction'][0])
    
    return time_points, predictions

if timeAnalysis:
    # Generate the time series data
    time_points, predictions = generate_time_series_predictions(
    test_input,
    test_input['eventTimesHistory'][0],
    duration_minutes=1)

    # Create a DataFrame for easier plotting
    df = pd.DataFrame({
        'timestamp': time_points,
        'prediction': predictions
    })

    # Create the plot
    plt.figure(figsize=(12, 6))
    plt.plot(df['timestamp'], df['prediction'], 'b-', linewidth=2)
    plt.xlabel('Time')
    plt.ylabel('Prediction')
    plt.title('Predictions Over Time')
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Show the plot
    plt.show()

    # Print some summary statistics
    print("\nSummary Statistics:")
    print(f"Mean prediction: {np.mean(predictions):.4f}")
    print(f"Standard deviation: {np.std(predictions):.4f}")
    print(f"Min prediction: {np.min(predictions):.4f}")
    print(f"Max prediction: {np.max(predictions):.4f}")

In [26]:
timeAnalysis=False

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import copy

def generate_time_series_predictions(base_input, start_time_str, 
                                     interval_seconds, num_intervals):
    # Parse the start time
    start_time = datetime.strptime(start_time_str, "%Y-%m-%dT%H:%M:%S.%fZ")
    
    # Generate time points
    time_points = [start_time + timedelta(seconds=i*interval_seconds) for i in range(num_intervals)]
    predictions = []
    
    # Run predictions for each time point
    for time_point in time_points:
        # Create a copy of the input
        modified_input = copy.deepcopy(base_input)
        # Update the eventTime
        modified_input['eventTimesHistory'][0] = time_point.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
        
        # Run inference
        results = run_inference(modified_input)
        
        # Based on the error output, results is a tuple where the first element is a DataFrame
        if isinstance(results, tuple) and len(results) > 0:
            # Extract the item prediction value (first row, first column)
            if isinstance(results[0], pd.DataFrame):
                prediction = results[0]['item_prediction'].values[0]
            else:
                prediction = results[0]
        else:
            # Fallback if the structure is different
            prediction = results['item_prediction'][0] if isinstance(results, dict) else results
            
        predictions.append(prediction)
    
    return time_points, predictions

if timeAnalysis:
    # Create figure with two subplots
    fig = plt.figure(figsize=(15, 6))
    
    try:
        # First plot - Seconds intervals
        time_points_seconds, predictions_seconds = generate_time_series_predictions(
            test_input,
            test_input['eventTimesHistory'][0],
            interval_seconds=5,  # 5 second intervals
            num_intervals=60    # 5 minutes total
        )
        
        # Convert timestamps to seconds elapsed
        seconds_elapsed = [(t - time_points_seconds[0]).total_seconds() for t in time_points_seconds]
        
        # Convert predictions to 1D array if needed
        predictions_seconds_array = np.array(predictions_seconds).flatten()
        
        ax1 = plt.subplot(1, 2, 1)
        ax1.plot(seconds_elapsed, predictions_seconds_array, 'b-', linewidth=2)
        ax1.set_xlabel('Seconds after the last event')
        ax1.set_ylabel('Prediction')
        ax1.set_title('Predictions Over Seconds')
        ax1.grid(True)
        
        # Second plot - 12-hour intervals
        time_points_days, predictions_days = generate_time_series_predictions(
            test_input,
            test_input['eventTimesHistory'][0],
            interval_seconds=12*3600,  # 12 hours in seconds
            num_intervals=120         # 60 days total
        )
        
        # Convert timestamps to days elapsed
        days_elapsed = [(t - time_points_days[0]).total_seconds() / (24*3600) for t in time_points_days]
        
        # Convert predictions to 1D array if needed
        predictions_days_array = np.array(predictions_days).flatten()
        
        ax2 = plt.subplot(1, 2, 2)
        ax2.plot(days_elapsed, predictions_days_array, 'r-', linewidth=2)
        ax2.set_xlabel('Days after the last event')
        ax2.set_ylabel('Prediction')
        ax2.set_title('Predictions Over Days')
        ax2.grid(True)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics with appropriate conversions
        print("\nSummary Statistics - Seconds:")
        print(f"Mean prediction: {np.mean(predictions_seconds_array):.4f}")
        print(f"Standard deviation: {np.std(predictions_seconds_array):.4f}")
        print(f"Min prediction: {np.min(predictions_seconds_array):.4f}")
        print(f"Max prediction: {np.max(predictions_seconds_array):.4f}")
        
        print("\nSummary Statistics - Days:")
        print(f"Mean prediction: {np.mean(predictions_days_array):.4f}")
        print(f"Standard deviation: {np.std(predictions_days_array):.4f}")
        print(f"Min prediction: {np.min(predictions_days_array):.4f}")
        print(f"Max prediction: {np.max(predictions_days_array):.4f}")
    
    except Exception as e:
        print(f"Error in time analysis: {str(e)}")
        
        # Print the structure of results to debug
        print("Debugging the results structure:")
        sample_result = run_inference(test_input)
        print(f"Type of result: {type(sample_result)}")
        if isinstance(sample_result, tuple) and len(sample_result) > 0:
            if isinstance(sample_result[0], pd.DataFrame):
                print("First element is a DataFrame with shape:", sample_result[0].shape)
                print("DataFrame columns:", sample_result[0].columns.tolist())
                print("First few rows of item_prediction column:")
                print(sample_result[0]['item_prediction'].head())
                # Get the actual value structure
                print("Structure of the item_prediction values:")
                print(type(sample_result[0]['item_prediction'].values))
                print(sample_result[0]['item_prediction'].values.shape)
        print(f"Content of result: {sample_result}")

In [27]:
#check output of test inference file
print(artifact_destination_dir)
import importlib.util
spec = importlib.util.spec_from_file_location("inference", artifact_destination_dir + "/inference.py")
inference_load = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_load)
if debugInputs:
    results, _ = inference_load.run_inference(single_input)
else:
    results = inference_load.run_inference(single_input)
    
results

../ModelImplementationWSDK/Math_student_proficiency_model/container/
All models, artifacts, and IP data loaded successfully.


,item_prediction,item_prediction_error,item_prediction_confidence,skill_prediction,skill_prediction_error,skill_prediction_confidence
0,0.184189,0.46866,6.252167,0.567032,0.561268,0.000806


In [28]:
#check output of test inference file (of bad type)
try:
    print(artifact_destination_dir)
    import importlib.util
    spec = importlib.util.spec_from_file_location("inference", artifact_destination_dir + "/inference.py")
    inference_load = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(inference_load)
    inference_load.run_inference(json.dumps(json.dumps(single_input)))
except Exception as e:
    print('successfully failed.')
    print('error: ', e)

../ModelImplementationWSDK/Math_student_proficiency_model/container/
All models, artifacts, and IP data loaded successfully.
successfully failed.
error:  Input data must be a dictionary for single inference or a list of dictionaries for batch inference


In [29]:
#check output of test inference file (of above input that should be working)
try:
    print(artifact_destination_dir)
    import importlib.util
    spec = importlib.util.spec_from_file_location("inference", artifact_destination_dir + "/inference.py")
    inference_load = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(inference_load)
    if debugInputs:
        response, _ = inference_load.run_inference(json.loads(json.dumps(single_input)))
    else:
        response = inference_load.run_inference(json.loads(json.dumps(single_input)))
        
    print('response: ', response)
except Exception as e:
    print('failed.')
    print('error: ', e)

../ModelImplementationWSDK/Math_student_proficiency_model/container/
All models, artifacts, and IP data loaded successfully.
response:     item_prediction  item_prediction_error  item_prediction_confidence  \
0         0.184189                0.46866                    6.252167   

   skill_prediction  skill_prediction_error  skill_prediction_confidence  
0          0.567032                0.561268                     0.000806  


In [30]:
#now test the flask container
# Get the Flask app from the inference module
app = inference_load.app

# Create a test client
client = app.test_client()

# Make a test request
response = client.post('/invocations', 
                      json=single_input,  # This will be automatically converted to JSON
                      content_type='application/json')

# Check the response
print(response.status_code)
print(response.get_json())

200
{'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 0.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 0.0, 'CORRECTNESS_LAG_4': 0.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 0.0, 'CORRECTNESS_LAG_7': 1.0, 'CORRECTNESS_LAG_8': 0.0, 'CORRECTNESS_LAG_9': 0.0, 'DURATIONSECONDS_LAG_1': 282.0, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': 266.0, 'DURATIONSECONDS_LAG_3': 23.0, 'DURATIONSECONDS_LAG_4': 187.0, 'DURATIONSECONDS_LAG_5': 120.0, 'DURATIONSECONDS_LAG_6': 157.0, 'DURATIONSECONDS_LAG_7': 179.0, 'DURATIONSECONDS_LAG_8': 220.0, 'DURATIONSECONDS_LAG_9': 211.0, 'LOG_DURATIONSECONDS_LAG_1': 5.641907070938114, 'LOG_DURATIONSECONDS_LAG_10': None, 'LOG_DURATIONSECONDS_LAG_2': 5.583496308781699, 'LOG_DURATIONSECONDS_LAG_3': 3.1354942159291497, 'LOG_DURATIONSECONDS_LAG_4': 5.231108616854587, 'LOG_DURATIONSECONDS_LAG_5': 4.787491742782046, 'LOG_DURATIONSECONDS_LAG_6': 5.056245805348308, 'LOG_DURATIONSECONDS_LAG_7': 5.18738580584075

In [31]:
#haskell check for nan/none outputs
def check_for_nans(response_body):
    return 'nan' in str(response_body)

result = response.get_json()
#result = "{'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 0.01, 'CORRECTNESS_LAG_10': 'nan', 'CORRECTNESS_LAG_2': 0.0}}}"

assert not check_for_nans(result), "ERROR Found NaN values in response - will cause Haskell server error"

In [32]:
# log_text = response.get_json()['debug_info']['feature_engineered_input']
log_text = response.get_json()['debug_info']
print(log_text)

{'feature_engineered_input': {'CORRECTNESS_LAG_1': 0.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': 0.0, 'CORRECTNESS_LAG_4': 0.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 0.0, 'CORRECTNESS_LAG_7': 1.0, 'CORRECTNESS_LAG_8': 0.0, 'CORRECTNESS_LAG_9': 0.0, 'DURATIONSECONDS_LAG_1': 282.0, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': 266.0, 'DURATIONSECONDS_LAG_3': 23.0, 'DURATIONSECONDS_LAG_4': 187.0, 'DURATIONSECONDS_LAG_5': 120.0, 'DURATIONSECONDS_LAG_6': 157.0, 'DURATIONSECONDS_LAG_7': 179.0, 'DURATIONSECONDS_LAG_8': 220.0, 'DURATIONSECONDS_LAG_9': 211.0, 'LOG_DURATIONSECONDS_LAG_1': 5.641907070938114, 'LOG_DURATIONSECONDS_LAG_10': None, 'LOG_DURATIONSECONDS_LAG_2': 5.583496308781699, 'LOG_DURATIONSECONDS_LAG_3': 3.1354942159291497, 'LOG_DURATIONSECONDS_LAG_4': 5.231108616854587, 'LOG_DURATIONSECONDS_LAG_5': 4.787491742782046, 'LOG_DURATIONSECONDS_LAG_6': 5.056245805348308, 'LOG_DURATIONSECONDS_LAG_7': 5.187385805840755, 'LOG_DURATIONSEC

In [33]:

import numpy as np

# Modify the comparison function to handle lists
import numpy as np
if debugInputs:
    for column in processed_inputs.columns:
        if column in log_dict:
            log_val = log_dict[column]
            df_val = processed_inputs[column].iloc[0]

            # Handle nan values first
            if isinstance(log_val, float) and isinstance(df_val, float):
                if np.isnan(log_val) and np.isnan(df_val):
                    continue  # Skip if both are nan
                elif np.isnan(log_val) or np.isnan(df_val):  # Only one is nan
                    print(f"Mismatch in {column}:")
                    print(f"Log value: {log_val}")
                    print(f"DataFrame value: {df_val}")
                    print()
                elif not np.isclose(log_val, df_val, rtol=1e-10):
                    print(f"Mismatch in {column}:")
                    print(f"Log value: {log_val}")
                    print(f"DataFrame value: {df_val}")
                    print()
            elif log_val != df_val:
                print(f"Mismatch in {column}:")
                print(f"Log value: {log_val}")
                print(f"DataFrame value: {df_val}")
                print()
                
print(f'{len(processed_inputs.columns)} columns checked.')

217 columns checked.


In [34]:
print([itemm['skillId'] for itemm in multiple_inputs])
multiple_inputs

['FAKE_WGYS3', 'FAKE_CTLGG', 'd6bdc51aa39fe3119503005056801da1', 'FAKE_B5K94', 'mohdiefo']


[{'skillId': 'FAKE_WGYS3',
  'questionId': 'question_548',
  'eventTime': '2025-05-27T21:24:38.579966',
  'questionIdsHistory': ['question_321',
   'question_847',
   'question_287',
   'question_536',
   'question_966',
   'question_64',
   'question_777',
   'question_329',
   'question_131',
   'question_156'],
  'correctnessHistory': [100, 100, 0, 100, 0, 100, 0, 0, 100, 100],
  'durationSecondsHistory': [267, 148, 100, 103, 231, 268, 76, 261, 243, 194],
  'eventTimesHistory': ['2025-05-27T21:24:38.579966',
   '2025-05-27T21:14:38.579966',
   '2025-05-27T21:04:38.579966',
   '2025-05-27T20:54:38.579966',
   '2025-05-27T20:44:38.579966',
   '2025-05-27T20:34:38.579966',
   '2025-05-27T20:24:38.579966',
   '2025-05-27T20:14:38.579966',
   '2025-05-27T20:04:38.579966',
   '2025-05-27T19:54:38.579966']},
 {'skillId': 'FAKE_CTLGG',
  'questionId': 'question_434',
  'eventTime': '2025-05-27T21:24:38.580039',
  'questionIdsHistory': ['question_101',
   'question_939',
   'question_676',
 

In [35]:
# Make a test request
response = client.post('/invocations', 
                      json=multiple_inputs,  # This will be automatically converted to JSON
                      content_type='application/json')

# Check the response
print(response.status_code)
print(response.get_json())

200
{'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 0.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 0.0, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': 0.0, 'CORRECTNESS_LAG_7': 0.0, 'CORRECTNESS_LAG_8': 1.0, 'CORRECTNESS_LAG_9': 1.0, 'DURATIONSECONDS_LAG_1': 148.0, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': 100.0, 'DURATIONSECONDS_LAG_3': 103.0, 'DURATIONSECONDS_LAG_4': 231.0, 'DURATIONSECONDS_LAG_5': 268.0, 'DURATIONSECONDS_LAG_6': 76.0, 'DURATIONSECONDS_LAG_7': 261.0, 'DURATIONSECONDS_LAG_8': 243.0, 'DURATIONSECONDS_LAG_9': 194.0, 'LOG_DURATIONSECONDS_LAG_1': 4.997212273764115, 'LOG_DURATIONSECONDS_LAG_10': None, 'LOG_DURATIONSECONDS_LAG_2': 4.605170185988091, 'LOG_DURATIONSECONDS_LAG_3': 4.634728988229636, 'LOG_DURATIONSECONDS_LAG_4': 5.442417710521793, 'LOG_DURATIONSECONDS_LAG_5': 5.5909869805108565, 'LOG_DURATIONSECONDS_LAG_6': 4.330733340286331, 'LOG_DURATIONSECONDS_LAG_7': 5.56452040732269

In [36]:
response.get_json()['prediction']

{'item_prediction': [0.26992762088775635,
  0.2821081280708313,
  0.21271361410617828,
  0.5167471170425415,
  0.16854482889175415],
 'item_prediction_confidence': [0.0008062110499286502,
  0.0008062110499286502,
  6.252166692196683,
  0.0008062110499286502,
  0.0008062110499286502],
 'item_prediction_error': [0.6187608242034912,
  0.5127192735671997,
  0.4809919595718384,
  0.5361812114715576,
  0.5325523018836975],
 'model_version': ['5.05', '5.05', '5.05', '5.05', '5.05'],
 'skill_prediction': [0.4421641230583191,
  0.4125671684741974,
  0.41627511382102966,
  0.44731074571609497,
  0.3415810763835907],
 'skill_prediction_confidence': [0.0008062110499286502,
  0.0008062110499286502,
  0.0008062110499286502,
  0.0008062110499286502,
  0.0008062110499286502],
 'skill_prediction_error': [0.5472745895385742,
  0.5112519860267639,
  0.6349669694900513,
  0.5576807260513306,
  0.5603089928627014]}

In [37]:
reproduced_message_2 = {"correctnessHistory": [0, 1, 0, 1, 0.75, 1], 
                        "durationSecondsHistory": [None, None, None, None, None, None], 
                        "eventTime": "2024-12-10T13:34:29.371403527Z", 
                        "eventTimesHistory": ["2024-12-10T13:34:29.371403527Z", "2024-12-10T13:34:20.931262655Z", None, None, None, None], "questionId": "6dc12d68-af7f-441f-9839-5882e115a2ca", 
                        "questionIdsHistory": ["6dc12d68-af7f-441f-9839-5882e115a2ca", "5c686c0c-d332-400e-b621-a8de2d0f824f", None, None, None, None], "skillId": "sqyudjgp"}


print('submitting: ', reproduced_message_2)
response = client.post('/invocations', 
                      json=reproduced_message_2,  # This will be automatically converted to JSON
                      content_type='application/json')

# Check the response
print(response.status_code)
print(response.get_json())

submitting:  {'correctnessHistory': [0, 1, 0, 1, 0.75, 1], 'durationSecondsHistory': [None, None, None, None, None, None], 'eventTime': '2024-12-10T13:34:29.371403527Z', 'eventTimesHistory': ['2024-12-10T13:34:29.371403527Z', '2024-12-10T13:34:20.931262655Z', None, None, None, None], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': ['6dc12d68-af7f-441f-9839-5882e115a2ca', '5c686c0c-d332-400e-b621-a8de2d0f824f', None, None, None, None], 'skillId': 'sqyudjgp'}
200
{'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 0.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 0.75, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATI

In [38]:
#check output of single iference test
results = run_inference(
    reproduced_message_2)
results

(   item_prediction  item_prediction_error  item_prediction_confidence  \
 0         0.463809               0.467195                    6.252167   
 
    skill_prediction  skill_prediction_error  skill_prediction_confidence  
 0          0.578482                0.461447                     6.313439  ,
    CORRECTNESS_LAG_1  CORRECTNESS_LAG_2  CORRECTNESS_LAG_3  CORRECTNESS_LAG_4  \
 0                1.0                0.0                1.0               0.75   
 
    CORRECTNESS_LAG_5  CORRECTNESS_LAG_6  CORRECTNESS_LAG_7  CORRECTNESS_LAG_8  \
 0                1.0                NaN                NaN                NaN   
 
    CORRECTNESS_LAG_9  CORRECTNESS_LAG_10  DURATIONSECONDS_LAG_1  \
 0                NaN                 NaN                    NaN   
 
    DURATIONSECONDS_LAG_2  DURATIONSECONDS_LAG_3  DURATIONSECONDS_LAG_4  \
 0                    NaN                    NaN                    NaN   
 
    DURATIONSECONDS_LAG_5  DURATIONSECONDS_LAG_6  DURATIONSECONDS_LAG_7  \


In [39]:
#test endpoint with empty time message
empty_test_message = {'skillId': 'ubuipdtp', 
                    'questionId': 'question_478', 
                    'eventTime': '',
                    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                                           'question_367', 
                                           'question_465', 'question_184', 'question_283', 'question_327'], 
                    'correctnessHistory': [], 
                    'durationSecondsHistory': [], 
                    'eventTimesHistory': []}


import boto3
import json

#check output of single iference test
results = run_inference(
    empty_test_message)
results

(   item_prediction  item_prediction_error  item_prediction_confidence  \
 0         0.425772               0.492371                    6.252167   
 
    skill_prediction  skill_prediction_error  skill_prediction_confidence  
 0          0.425772                0.492371                     6.252167  ,
    CORRECTNESS_LAG_1  CORRECTNESS_LAG_2  CORRECTNESS_LAG_3  CORRECTNESS_LAG_4  \
 0                NaN                NaN                NaN                NaN   
 
    CORRECTNESS_LAG_5  CORRECTNESS_LAG_6  CORRECTNESS_LAG_7  CORRECTNESS_LAG_8  \
 0                NaN                NaN                NaN                NaN   
 
    CORRECTNESS_LAG_9  CORRECTNESS_LAG_10  DURATIONSECONDS_LAG_1  \
 0                NaN                 NaN                    NaN   
 
    DURATIONSECONDS_LAG_2  DURATIONSECONDS_LAG_3  DURATIONSECONDS_LAG_4  \
 0                    NaN                    NaN                    NaN   
 
    DURATIONSECONDS_LAG_5  DURATIONSECONDS_LAG_6  DURATIONSECONDS_LAG_7  \


In [40]:
#test new 404 error
client = app.test_client()

# Make a request to a non-existent endpoint
response = client.post('/nonexistent_endpoint', 
                     json={'some': 'data'},
                     content_type='application/json')

print('successfully failed.') if response.status_code == 404 else print(f'unexpected status code: {response.status_code}')

# Check status code
print("Status code:", response.status_code)

# Print response data
print("Response:", response.get_json())

successfully failed.
Status code: 404
Response: {'error': 'Resource not found', 'error_details': {'error_type': '404', 'error_message': 'The requested URL was not found on the server.'}}


In [41]:
#test empty
import json

# Test empty message
empty_test_message = {
    'skillId': 'epwxqpjn', 
    'questionId': 'question_478', 
    'eventTime': '',
    'questionIdsHistory': [], 
    'correctnessHistory': [], 
    'durationSecondsHistory': [], 
    'eventTimesHistory': []
}

# # Run inference with empty message
# results = run_inference(empty_test_message)
# print(results)

# Test non-existent endpoint
client = app.test_client()
response = client.post('/invocations', 
                     json=empty_test_message,
                     content_type='application/json')
response.get_json()

{'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': None,
   'CORRECTNESS_LAG_10': None,
   'CORRECTNESS_LAG_2': None,
   'CORRECTNESS_LAG_3': None,
   'CORRECTNESS_LAG_4': None,
   'CORRECTNESS_LAG_5': None,
   'CORRECTNESS_LAG_6': None,
   'CORRECTNESS_LAG_7': None,
   'CORRECTNESS_LAG_8': None,
   'CORRECTNESS_LAG_9': None,
   'DURATIONSECONDS_LAG_1': None,
   'DURATIONSECONDS_LAG_10': None,
   'DURATIONSECONDS_LAG_2': None,
   'DURATIONSECONDS_LAG_3': None,
   'DURATIONSECONDS_LAG_4': None,
   'DURATIONSECONDS_LAG_5': None,
   'DURATIONSECONDS_LAG_6': None,
   'DURATIONSECONDS_LAG_7': None,
   'DURATIONSECONDS_LAG_8': None,
   'DURATIONSECONDS_LAG_9': None,
   'LOG_DURATIONSECONDS_LAG_1': None,
   'LOG_DURATIONSECONDS_LAG_10': None,
   'LOG_DURATIONSECONDS_LAG_2': None,
   'LOG_DURATIONSECONDS_LAG_3': None,
   'LOG_DURATIONSECONDS_LAG_4': None,
   'LOG_DURATIONSECONDS_LAG_5': None,
   'LOG_DURATIONSECONDS_LAG_6': None,
   'LOG_DURATIONSECONDS_LAG_7': None,
   'LOG_DUR